# Stage 2: predicting crash severity, and whether the safety score helps

**Two questions**

1. How well can we predict whether a cyclist crash is serious or fatal,
   using only information that existed before the crash?
2. Does the perceived safety score from the first paper make that
   prediction better?

**Two models**

- **Street model**: the place and the conditions (road, junction, weather,
  nearby places, crash history of the area).
- **Full model**: the street model plus the cyclist's age, sex and age group.

The safety score is added on top of each one, so there are
13 versions in total: street and full, each without and with the score in four forms, two references (safety score only, cyclist only) and the full model with photo timing.

**Data:** crashes from 1 November 2016 onwards. From that month the
Metropolitan Police record injuries and the severity is derived from them,
so every crash in this notebook uses the same definition of "serious".

**How the notebook is organised**

| Section | What it does |
|---|---|
| 1. Setup | Settings, paths, one chart style, the feature dictionary |
| 2. Data | Crashes, the time split (70% / 15% / 15%), crashes by month and season, the older-crash adjustment |
| 3. Features | Candidate features, missing values, association with severity, selection options |
| 4. Model search | XGBoost, elastic-net logistic regression and LightGBM, 100 settings each, with 10 feature-selection options |
| 5. Final model | Four candidates (the three models and their average), the test of older crashes, every version on validation |
| 6. Results | Every version on the test part, the models for the paper, photo timing and the safety score, charts |
| 7. Feature impact | SHAP: size and direction of each input's effect, with readable names |
| 8. Severity breakdowns and maps | Share of crashes and serious or fatal share by group and by the top features; hexagon and borough maps |
| 9. Robustness | The method repeated on two time periods |
| 10. Outputs | Feature journey, feature dictionary, decision log, data files, saved models, Excel file and the summary |

**Fair by design:** every choice (features, model type, settings,
cut-off) is made by a rule fixed in advance, using only the fitting and
validation parts. The test part is used once, in Section 6.

**Saved** in `/kaggle/working/stage2_outputs`: `stage2_results.xlsx`
(every table and the data behind every chart), `feature_dictionary.xlsx`,
the `charts` folder (every chart as PDF and as 300 dpi PNG),
`eligible_pre_crash_inputs.csv`, `crashes_for_breakdowns.csv`,
`test_crashes_with_predictions.csv`, `shap_values_test.csv`,
`test_predictions.csv` and `final_models.joblib`.

## Section 1: setup

In [ ]:
import sys, subprocess, importlib.util
for module, pip_name in {"xgboost": "xgboost", "lightgbm": "lightgbm", "openpyxl": "openpyxl"}.items():
    if importlib.util.find_spec(module) is None:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], capture_output=True, text=True)
        print(f"{module}: {'installed' if r.returncode == 0 else 'could not be installed'}")
    else:
        print(f"{module}: already installed")
print(f"geopandas (for the borough maps): {'available' if importlib.util.find_spec('geopandas') else 'not available, maps will be skipped'}")

In [ ]:
import os, re, json, time, platform, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import rankdata, randint, uniform, loguniform, spearmanr
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)

# ---------------- paths ----------------
INPUT_FOLDER = "..."
FEATURES_CSV = os.path.join(INPUT_FOLDER, "collisions_final_features.csv")
RAW_CSV = os.path.join(INPUT_FOLDER, "collisions_final.csv")              # for the severity adjustment of older crashes
CATALOG_CSV = os.path.join(INPUT_FOLDER, "feature_catalog.csv")
JOURNEY_XLSX = os.path.join(INPUT_FOLDER, "feature_journey.xlsx")
BOROUGHS_GPKG = os.path.join(INPUT_FOLDER, "London_Boroughs.gpkg")      # optional, for the maps
OUTPUT_FOLDER = "/kaggle/working/stage2_outputs"
CHART_FOLDER = os.path.join(OUTPUT_FOLDER, "charts")
os.makedirs(CHART_FOLDER, exist_ok=True)
EXCEL_PATH = os.path.join(OUTPUT_FOLDER, "stage2_results.xlsx")

# ---------------- data ----------------
OUTCOME_COLUMN = "casualty_severity"
START_DATE = "2016-11-01"
RANDOM_SEED = 101
FIT_SHARE, VALIDATION_SHARE = 0.70, 0.15            # the test part is the remaining 15%
EXCLUDED_COLUMNS = {
    "exposure_site_id": "ID number of a cycle-counting site: a label, not a measurement",
    "exposure_match_level": "Describes how the exposure value was matched: data processing, not the street",
    "roadno1": "Road number (for example A406): names one specific road, works like a location label",
    "roadno2": "Road number of the second road: names one specific road, works like a location label",
}
# A "missing" marker is kept only where missing says something about the place.
PLACE_FAMILIES = {"Points of interest", "Cyclist exposure", "Noise", "Deprivation and high street",
                  "Crash history counts", "Weather readings"}

# ---------------- feature selection options ----------------
P_VALUE_LEVELS = {"Correlation p<0.05": 1.960, "Correlation p<0.01": 2.576, "Correlation p<0.001": 3.291}
FIXED_CORRELATION = {"Correlation >= 0.03": 0.03}
MRMR_SIZES = [20, 40, 80, 140]
# Older crashes (before the start date) can be used for TRAINING ONLY, with the Department for Transport's
# adjustment: each one counts as serious with weight p and as slight with weight 1 - p, where p is the
# probability that it would have been recorded as serious under injury-based recording. Validation and
# test always use crashes from the start date only. Used only if it raises validation balanced accuracy.
OLDER_MIN_GAIN = 0.0                  # points of balanced accuracy, and ROC-AUC must not fall
STABLE_OPTION = "Stable p<0.01 (3 of 5 halves)"
OPTIONS = list(P_VALUE_LEVELS) + list(FIXED_CORRELATION) + [STABLE_OPTION] + [f"mRMR {k}" for k in MRMR_SIZES] + ["All features"]
MIN_FEATURES = 5                      # a threshold that keeps fewer keeps the 5 strongest instead
CHANCE_SHUFFLES = 3

# ---------------- model search ----------------
MODEL_TYPES = ["XGBoost", "Elastic-net logistic regression", "LightGBM"]
AVERAGE = "Average of the three"         # a fourth candidate: the average of the three best models
N_SETTINGS = 100                      # per model type
N_FOLDS = 5
SEEDS = [101, 202, 303, 404, 505]     # tree models are averaged over these seeds in the final training

# ---------------- evaluation ----------------
BOOTSTRAP_RANGES = 500
BOOTSTRAP_DIFFERENCES = 1000
BOOTSTRAP_GROUPS = 300
MIN_GROUP = {"crashes": 200, "serious": 20, "slight": 20}
N_CPU = os.cpu_count() or 4

for path in (FEATURES_CSV, CATALOG_CSV, JOURNEY_XLSX):
    assert os.path.exists(path), f"Not found: {path}. Check that the file is in the Kaggle dataset."
notebook_start = time.time()
print(f"Settings loaded. CPU cores: {N_CPU}. Borough file: {'found' if os.path.exists(BOROUGHS_GPKG) else 'not found, maps will be skipped'}")
print(f"Search: {N_SETTINGS} settings x {len(MODEL_TYPES)} model types, each on {N_FOLDS} time-ordered folds, "
      f"with {len(OPTIONS)} feature-selection options")

In [ ]:
SHEETS = {}

def save_excel():
    with pd.ExcelWriter(EXCEL_PATH) as writer:
        for name, table in SHEETS.items():
            table.to_excel(writer, sheet_name=name[:31], index=False)
    print(f"Excel updated: {len(SHEETS)} sheets.")

def save_chart(fig, name):
    for ext, kw in [("pdf", {}), ("png", {"dpi": 300})]:
        fig.savefig(os.path.join(CHART_FOLDER, f"{name}.{ext}"), bbox_inches="tight", **kw)
    plt.show()
    plt.close(fig)
    print(f"Chart saved: {name}.pdf and {name}.png")

def show(table, title=None, digits=4):
    if title:
        print(title)
    display(table.round(digits) if isinstance(table, pd.DataFrame) else table)

# ---------------- one chart style for every figure (journal widths, colour-blind-safe colours) ----------------
SINGLE, DOUBLE = 3.35, 6.9          # inches: one journal column, two columns
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.family": "DejaVu Sans", "font.size": 8,
                     "axes.titlesize": 9, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 8,
                     "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5, "legend.frameon": False,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
                     "xtick.major.width": 0.6, "ytick.major.width": 0.6, "axes.grid": False,
                     "pdf.fonttype": 42, "ps.fonttype": 42, "figure.constrained_layout.use": False})
BLUE, LIGHT, ORANGE, GREEN, GREY = "#0F4C81", "#8FB3D3", "#D55E00", "#009E73", "#7A7A7A"
RED = ORANGE
RAISES, LOWERS = "#B2182B", "#2166AC"
SEASON_COLOURS = {"Winter": "#2166AC", "Spring": "#009E73", "Summer": "#E69F00", "Autumn": "#B2182B"}
MAP_CMAP = "cividis"

def wilson(k, n, z=1.96):
    """95% range of a share (Wilson method), in %."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h) * 100, (c + h) * 100

# ---------------- feature dictionary: readable names for codes ----------------
CATEGORY_WORDS = {
    "Single Cwy": "Single carriageway", "Dual Cwy": "Dual carriageway", "One Way St": "One-way street", "Rbt": "Roundabout",
    "Slip Road": "Slip road", "No Jun In 20M": "No junction within 20 m", "T/Stag Jun": "T or staggered junction",
    "Crossroads": "Crossroads", "Roundabout": "Roundabout", "Mini": "Mini roundabout", "Multi Jun": "Multiple junction",
    "Priv Drive": "Private drive or entrance", "Other Jun": "Other junction", "Give Way/Uncontrolled": "Give way or uncontrolled",
    "Auto Sig": "Automatic traffic signal", "Stop Sign": "Stop sign", "Auth Person": "Authorised person",
    "Not at junction": "Not at a junction", "No Xing Facility In 50M": "No crossing within 50 m",
    "Pelican Or Similar": "Pelican, puffin, toucan or similar", "Zebra": "Zebra crossing", "Road-Dry": "Dry", "Road-Wet": "Wet or damp",
    "Road-Frost/Ice": "Frost or ice", "Road-Snow": "Snow", "Road-Flood": "Flood", "Daylight": "Daylight", "Dark": "Darkness",
    "20 and under": "20 mph or less", "missing": "missing", "nan": "missing",
}
FEATURE_NAMES = {
    "borough": "Borough", "highway": "Highway authority", "roadclass1": "Road class", "roadclass2": "Class of second road",
    "roadtype": "Road type", "speedlimit": "Speed limit (mph)", "junctiondetail": "Junction type", "junctioncontrol": "Junction control",
    "pedcrossingdecoded": "Pedestrian crossing", "lightconditionsbanded": "Light conditions", "lightconditions_collapsed": "Light conditions (grouped)",
    "weather": "Weather (police record)", "roadsurface": "Road surface", "specialconditions": "Special conditions at site",
    "grade_percent": "Road gradient (%)", "road_avg_width": "Road width, average (m)", "road_min_width": "Road width, minimum (m)",
    "streetlight_coverage": "Street-light coverage", "pavement_avg_width_m": "Pavement width, average (m)",
    "pavement_min_width_m": "Pavement width, minimum (m)", "road_route_hierarchy": "Road route hierarchy", "bank_holiday": "Bank holiday",
    "IMDScore": "Deprivation score (IMD)", "IMDRank": "Deprivation rank (IMD)", "IMDDecile": "Deprivation decile (1 = most deprived)", "HS_bin": "On a high street",
    "noise_lden_yr": "Road noise, day-evening-night (dB)", "noise_lnight_yr": "Road noise at night (dB)", "noise_lday": "Road noise, daytime (dB)",
    "noise_lnight": "Road noise at night, 2017 map (dB)", "noise_lden_yr_band": "Road noise band, day-evening-night",
    "noise_lnight_yr_band": "Road noise band, night", "exposure_avg_cyclists": "Cyclist flow (average count)",
    "exposure_adjusted_collision_history": "Past crashes relative to cyclist flow",
    "hotspot_grad_1y_100m__5y_500m": "Crash hotspot gradient (last year within 100 m vs 5 years within 500 m)",
    "hotspot_grad_inv_1y_100m__5y_500m": "Crash hotspot gradient, inverse",
    "nearest_school_crow_type": "Type of nearest school", "nearest_school_road_phase": "Phase of nearest school",
    "date_month": "Month", "date_dayofweek": "Day of the week", "date_hour": "Hour of the day", "date_is_weekend": "Weekend",
    "date_is_rush_hour": "Rush hour", "date_is_night": "Night", "date_season": "Season", "date_hour_bin": "Time of day",
    "age": "Cyclist age (years)", "casualty_sex": "Cyclist sex", "age_group": "Cyclist age group",
    "safety_score_heading_0": "Perceived safety, photo facing 0 degrees", "safety_score_heading_90": "Perceived safety, photo facing 90 degrees",
    "safety_score_heading_180": "Perceived safety, photo facing 180 degrees", "safety_score_heading_270": "Perceived safety, photo facing 270 degrees",
    "safety_score_mean": "Perceived safety, average of 4 photos", "safety_score_std": "Perceived safety, spread across 4 photos",
    "safety_score_min": "Perceived safety, lowest of 4 photos", "safety_score_expected": "Perceived safety, expected class",
    "photo_gap_months": "Months between crash and street photo",
}
PAIR_PARTS = {"cx_borough_roadclass1": ("Borough", "road class"), "cx_borough_roadtype": ("Borough", "road type"),
              "cx_borough_speed": ("Borough", "speed limit"), "cx_roadtype_speed": ("Road type", "speed limit"),
              "cx_roadclass_speed": ("Road class", "speed limit"), "cx_junctiondetail_speed": ("Junction type", "speed limit"),
              "cx_junctioncontrol_speed": ("Junction control", "speed limit"), "cx_junction_roadtype": ("Junction type", "road type"),
              "cx_light_hourbin": ("Light conditions", "time of day"), "cx_light_roadtype": ("Light conditions", "road type"),
              "cx_pedcross_junction": ("Pedestrian crossing", "junction type")}
POI = {"school": "school", "pub": "pub", "overground": "Overground station", "underground": "Underground station",
       "signal": "traffic signal", "bus_stop": "bus stop", "bus_stops": "bus stops", "schools": "schools", "pubs": "pubs", "signals": "traffic signals"}
WEATHER = {"temperature_2m": "Air temperature", "relativehumidity_2m": "Relative humidity", "dewpoint_2m": "Dew point",
           "apparent_temperature": "Feels-like temperature", "precipitation": "Precipitation", "rain": "Rain", "snowfall": "Snowfall",
           "snow_depth": "Snow depth", "weathercode": "Weather code", "pressure_msl": "Sea-level pressure", "surface_pressure": "Surface pressure",
           "cloudcover": "Cloud cover", "cloudcover_low": "Low cloud cover", "cloudcover_mid": "Mid cloud cover", "cloudcover_high": "High cloud cover",
           "windspeed_10m": "Wind speed at 10 m", "windgusts_10m": "Wind gusts at 10 m", "windspeed_100m": "Wind speed at 100 m",
           "winddirection_10m": "Wind direction at 10 m", "winddirection_100m": "Wind direction at 100 m",
           "et0_fao_evapotranspiration": "Evapotranspiration", "vapour_pressure_deficit": "Vapour pressure deficit"}

def readable_feature(f):
    if f in FEATURE_NAMES:
        return FEATURE_NAMES[f]
    if f in PAIR_PARTS:
        return f"{PAIR_PARTS[f][0]} x {PAIR_PARTS[f][1]}"
    if f.startswith("prior_ratio_"):
        return f"Earlier serious share for the same {readable_feature(f[len('prior_ratio_'):]).lower()}"
    m = re.fullmatch(r"past_accidents_(\d+)m_(\d+)y(?:_(slight|serious|fatal))?", f)
    if m:
        kind = {"slight": "slight crashes", "serious": "serious crashes", "fatal": "fatal crashes", None: "crashes"}[m.group(3)]
        return f"Past {kind} within {m.group(1)} m, last {m.group(2)} year{'s' if m.group(2) != '1' else ''}"
    m = re.fullmatch(r"nearest_(\w+?)_(road|crow)_m", f)
    if m:
        return f"Distance to nearest {POI.get(m.group(1), m.group(1).replace('_', ' '))}, {'by road' if m.group(2) == 'road' else 'straight line'} (m)"
    m = re.fullmatch(r"n_(\w+)_100m", f)
    if m:
        return f"Number of {POI.get(m.group(1), m.group(1).replace('_', ' '))} within 100 m"
    m = re.fullmatch(r"archive\.hourly\.(.+?)(?:_lag_(\d+)h)?(_sin|_cos)?", f)
    if m:
        base = m.group(1)
        sincos = m.group(3)
        if base.endswith(("_sin", "_cos")):
            sincos, base = "_" + base[-3:], base[:-4]
        soil = re.fullmatch(r"soil_(temperature|moisture)_(\d+)_to_(\d+)cm", base)
        name = f"Soil {soil.group(1)}, {soil.group(2)} to {soil.group(3)} cm" if soil else WEATHER.get(base, base.replace("_", " ").capitalize())
        if sincos:
            name += f" ({'sine' if sincos == '_sin' else 'cosine'})"
        return name + (f", {m.group(2)} h before" if m.group(2) else "")
    return f.replace("_", " ").capitalize()

def readable_category(v):
    v = str(v)
    if "__" in v:
        return " x ".join(readable_category(p) for p in v.split("__"))
    return CATEGORY_WORDS.get(v, v)

def readable_input(nice):
    """Readable label for a model input: 'feature = category', 'feature: missing' or a number feature."""
    if " = " in nice:
        f, c = nice.split(" = ", 1)
        return f"{readable_feature(f)}: {readable_category(c)}"
    if nice.endswith(": missing"):
        return f"{readable_feature(nice[:-9])}: missing"
    return readable_feature(re.sub(r" \(\d+\)$", "", nice))

print("Helpers ready.")

## Section 2: data and the time split

Crashes from 1 November 2016, without an outcome, a date or a safety score
left out. They are put in date order and split into:

- **Fitting** (oldest 70%): to select features, search and train.
- **Validation** (next 15%): to compare model types and choose the cut-off.
- **Test** (newest 15%): used once, in Section 6.

The final model is trained on fitting + validation (85% of the crashes).

In [ ]:
data = pd.read_csv(FEATURES_CSV, low_memory=False, parse_dates=["accident_dt"], dtype={"collision_id": str})
catalog = pd.read_csv(CATALOG_CSV)
rows_in_file = len(data)
data["target"] = data[OUTCOME_COLUMN].map({"Slight": 0, "Serious": 1, "Fatal": 1})

# Junction control is empty when the crash is not at a junction: there is nothing to control.
# Those blanks become "Not at junction", and the pair and the earlier-severity ratio are rebuilt.
NOT_AT_JUNCTION = "Not at junction"
no_junction = data["junctiondetail"].astype(object).fillna("").astype(str).str.lower().str.contains("no jun|not at")
structural = no_junction & data["junctioncontrol"].isna()
data["junctioncontrol"] = data["junctioncontrol"].astype(object)
data.loc[structural, "junctioncontrol"] = NOT_AT_JUNCTION
if "cx_junctioncontrol_speed" in data.columns:
    speed_part = data["cx_junctioncontrol_speed"].astype(object).fillna("nan").astype(str).str.split("__").str[-1]
    data["cx_junctioncontrol_speed"] = data["cx_junctioncontrol_speed"].astype(object)
    data.loc[structural, "cx_junctioncontrol_speed"] = NOT_AT_JUNCTION + "__" + speed_part[structural]

def earlier_ratio(col, strength=20):
    """Serious share of the crash's category among EARLIER crashes (all years), smoothed towards the overall
    share at that moment, divided by that overall share. Crashes at the same time do not count each other."""
    d = data[["accident_dt", col, "target"]].sort_values("accident_dt", kind="mergesort")
    valid = d["target"].notna().astype(float)
    t = d["target"].fillna(0) * valid
    key = d[col].astype(object).fillna("__missing__").astype(str)
    n_all, s_all = valid.cumsum() - valid, t.cumsum() - t
    n_cat, s_cat = valid.groupby(key).cumsum() - valid, t.groupby(key).cumsum() - t
    n_all, s_all = n_all.groupby(d["accident_dt"]).transform("first"), s_all.groupby(d["accident_dt"]).transform("first")
    n_cat = n_cat.groupby([d["accident_dt"], key]).transform("first")
    s_cat = s_cat.groupby([d["accident_dt"], key]).transform("first")
    p0 = (s_all / n_all.replace(0, np.nan))
    ratio = ((s_cat + strength * p0) / (n_cat + strength)) / p0
    ratio[key == "__missing__"] = np.nan
    return ratio.reindex(data.index)

if "prior_ratio_junctioncontrol" in data.columns:
    data["prior_ratio_junctioncontrol"] = earlier_ratio("junctioncontrol")
recent = data["accident_dt"] >= START_DATE
print(f"Junction control set to '{NOT_AT_JUNCTION}': {int((structural & recent).sum()):,} crashes from {START_DATE} "
      f"(still empty after this: {int((data['junctioncontrol'].isna() & recent).sum()):,}).")
ped_missing = data["pedcrossingdecoded"].isna() & recent
print(f"Pedestrian crossing empty: {int(ped_missing.sum()):,} crashes; share at a junction vs not: "
      f"{ped_missing[recent & ~no_junction].mean() * 100:.1f}% vs {ped_missing[recent & no_junction].mean() * 100:.1f}%. "
      "A similar share in both groups means the blanks are not structural, so they follow rule 4.")
before_start = (data["accident_dt"] < START_DATE).sum()
usable = data["target"].notna() & data["accident_dt"].notna() & data["safety_score_mean"].notna()
keep = usable & (data["accident_dt"] >= START_DATE)
no_score = (data["accident_dt"] >= START_DATE) & data["target"].notna() & data["safety_score_mean"].isna()
older = data[usable & (data["accident_dt"] < START_DATE)].copy()
data = data[keep].copy()
data["target"] = data["target"].astype(int)
data = data.sort_values("accident_dt", kind="mergesort").reset_index(drop=True)
n = len(data)

# Probability that each older crash would be serious or fatal under injury-based recording.
# Per cyclist casualty: 1 if fatal, otherwise the DfT adjusted probability (or the recorded label if missing).
# Per crash: the chance that at least one cyclist is serious or fatal, 1 - product of (1 - p).
OLDER_OK = False
if os.path.exists(RAW_CSV) and len(older):
    raw = pd.read_csv(RAW_CSV, usecols=lambda c: c in ("collision_id", "casualty_severity", "casualty_adj_serious"),
                      dtype={"collision_id": str}, low_memory=False)
    if {"collision_id", "casualty_severity", "casualty_adj_serious"} <= set(raw.columns):
        sev = raw["casualty_severity"].astype(str)
        p_cas = pd.to_numeric(raw["casualty_adj_serious"], errors="coerce")
        p_cas = p_cas.where(p_cas.notna(), (sev == "Serious").astype(float))
        p_cas[sev == "Fatal"] = 1.0
        p_crash = (1 - (1 - p_cas.clip(0, 1)).groupby(raw["collision_id"]).prod())
        older["p_serious"] = older["collision_id"].map(p_crash)
        older["p_serious"] = older["p_serious"].fillna(older["target"])
        older = older.sort_values("accident_dt", kind="mergesort")
        older["target"] = older["target"].astype(int)
        OLDER_OK = True
        print(f"Older crashes available for training: {len(older):,} ({older['accident_dt'].min():%Y-%m} to {older['accident_dt'].max():%Y-%m}). "
              f"Recorded serious or fatal: {older['target'].mean() * 100:.1f}%; after the adjustment: {older['p_serious'].mean() * 100:.1f}%.")
    else:
        print("The raw file has no adjustment column: older crashes are not used.")
else:
    print(f"{RAW_CSV} not found: older crashes are not used.")

fit_end, val_end = int(n * FIT_SHARE), int(n * (FIT_SHARE + VALIDATION_SHARE))
IDX = {"fit": np.arange(0, fit_end), "val": np.arange(fit_end, val_end), "test": np.arange(val_end, n)}
IDX["fit+val"] = np.concatenate([IDX["fit"], IDX["val"]])
IDX["val+test"] = np.concatenate([IDX["val"], IDX["test"]])
half = len(IDX["val"]) // 2
IDX["val1"], IDX["val2"] = IDX["val"][:half], IDX["val"][half:]

# The older crashes are added at the end of the table, after the split, so they are never part of
# fitting, validation or test. They are only added to training when the rule in Section 5.2 says so.
if OLDER_OK:
    data = pd.concat([data, older.drop(columns="p_serious")], ignore_index=True)
    IDX["older"] = np.arange(n, len(data))
    OLDER_P = older["p_serious"].values
else:
    IDX["older"], OLDER_P = np.arange(0), np.zeros(0)
y_all = data["target"].values
yv, yt = y_all[IDX["val"]], y_all[IDX["test"]]

def describe(name, idx):
    part = data.iloc[idx]
    return {"Part": name, "Share": f"{len(idx) / n * 100:.1f}%", "Crashes": len(part),
            "From": part["accident_dt"].min().strftime("%Y-%m-%d"), "To": part["accident_dt"].max().strftime("%Y-%m-%d"),
            "Serious or fatal %": round(part["target"].mean() * 100, 1)}

split_table = pd.DataFrame([describe("Fitting", IDX["fit"]), describe("Validation", IDX["val"]), describe("Test", IDX["test"])])
print(f"Crashes in the file: {rows_in_file:,}. Before {START_DATE}: {before_start:,}. "
      f"From {START_DATE} without a safety score: {int(no_score.sum())}. Used: {n:,}.")
show(split_table)
SHEETS["Data and split"] = split_table

recent = data.iloc[:n].copy()
recent["month"] = recent["accident_dt"].dt.to_period("M").dt.to_timestamp()
season_of = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
             6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
recent["season"] = recent["accident_dt"].dt.month.map(season_of)
monthly = recent.groupby("month").agg(crashes=("target", "size"), serious=("target", "mean"), season=("season", "first")).reset_index()
fig, axes = plt.subplots(2, 1, figsize=(DOUBLE, 4.4), sharex=True, gridspec_kw={"height_ratios": [1.3, 1]})
for name, idx, colour in [("Fitting", IDX["fit"], "#EEF2F6"), ("Validation", IDX["val"], "#FBF0DC"), ("Test", IDX["test"], "#E6F2EC")]:
    d = data.iloc[idx]["accident_dt"]
    for ax in axes:
        ax.axvspan(d.min(), d.max(), color=colour, linewidth=0, zorder=0)
    axes[0].text(d.min() + (d.max() - d.min()) / 2, 1.02, name, transform=axes[0].get_xaxis_transform(), ha="center", fontsize=7.5, color=GREY)
axes[0].bar(monthly["month"], monthly["crashes"], width=25, color=[SEASON_COLOURS[x] for x in monthly["season"]], zorder=2)
axes[0].set_ylabel("Crashes per month")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in SEASON_COLOURS.values()]
axes[0].legend(handles, SEASON_COLOURS.keys(), ncol=4, loc="upper left", bbox_to_anchor=(0, 0.98))
axes[1].plot(monthly["month"], monthly["serious"] * 100, color=BLUE, linewidth=1, zorder=2)
axes[1].set_ylabel("Serious or fatal (%)\nout of the month's crashes")
fig.suptitle("Cyclist crashes per month by season, the serious or fatal share, and the time split", x=0.01, ha="left", fontsize=9, fontweight="bold")
plt.tight_layout()
save_chart(fig, "01_crashes_by_month_and_season")

by_month = recent.groupby(recent["accident_dt"].dt.month).agg(crashes=("target", "size"), serious=("target", "sum"))
by_month["crashes per year"] = by_month["crashes"] / recent["accident_dt"].dt.year.nunique()
by_month["serious %"] = by_month["serious"] / by_month["crashes"] * 100
season_table = recent.groupby("season").agg(crashes=("target", "size"), serious=("target", "sum"))
season_table["share of all crashes %"] = season_table["crashes"] / season_table["crashes"].sum() * 100
season_table["serious %"] = season_table["serious"] / season_table["crashes"] * 100
season_parts = pd.crosstab(recent["season"], pd.Series(np.where(np.isin(np.arange(n), IDX["fit"]), "Fitting",
                           np.where(np.isin(np.arange(n), IDX["val"]), "Validation", "Test")), index=recent.index), normalize="columns") * 100
show(season_table.reindex(list(SEASON_COLOURS)).reset_index(), "Crashes by season (all crashes from the start date):", 1)
show(season_parts.reindex(list(SEASON_COLOURS)).round(1).reset_index(), "\nShare of each part's crashes by season (%):", 1)
SHEETS["Chart data months"] = monthly.assign(month=monthly["month"].dt.strftime("%Y-%m"), serious=monthly["serious"] * 100).rename(columns={"serious": "serious %"})
SHEETS["Seasons"] = season_table.reset_index().round(3)

## Section 3: features

### 3.1 Candidate features

The first notebook produced 248 candidate street features. Four more are
excluded here because they are not properties of the kind of street (an
ID, a processing detail, and two road numbers that name one specific
road). The full model adds the cyclist's age, sex and age group.

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import average_precision_score, roc_curve
import sklearn

CANDIDATES = catalog.loc[catalog["group"].isin(["Environment", "Engineered"]), "feature"].tolist()
BASE_ALL = [c for c in CANDIDATES if c not in EXCLUDED_COLUMNS]
SAFETY_COLS = catalog.loc[catalog["group"] == "Safety score", "feature"].tolist()
for c in BASE_ALL:
    if not pd.api.types.is_numeric_dtype(data[c]):
        data[c] = data[c].where(data[c].isna(), data[c].astype(str))
data["age"] = pd.to_numeric(data["age"], errors="coerce")
data.loc[(data["age"] < 1) | (data["age"] > 100), "age"] = np.nan
data["casualty_sex"] = data["casualty_sex"].where(data["casualty_sex"].isna(), data["casualty_sex"].astype(str))
data["age_group"] = pd.cut(data["age"], bins=[0, 15, 24, 34, 44, 54, 64, 120],
                           labels=["under 16", "16-24", "25-34", "35-44", "45-54", "55-64", "65+"]).astype(str)
data.loc[data["age"].isna(), "age_group"] = np.nan
CYCLIST = ["age", "casualty_sex", "age_group"]
FAMILY = catalog.set_index("feature")["family"].to_dict()
FAMILY.update({c: "Cyclist" for c in CYCLIST})

# Photo timing: months between the crash and the street photo (positive = photo taken after the crash).
# Tested as one extra version only (Section 6): it describes data collection rather than the street, and it can
# carry the crash date, so it is not part of the eligible street features.
PHOTO = []
if "streetview_time_diff_days" not in data.columns and os.path.exists(RAW_CSV):
    gap = pd.read_csv(RAW_CSV, usecols=lambda c: c in ("collision_id", "streetview_time_diff_days"), dtype={"collision_id": str}, low_memory=False)
    if "streetview_time_diff_days" in gap.columns:
        data["streetview_time_diff_days"] = data["collision_id"].map(gap.groupby("collision_id")["streetview_time_diff_days"].mean())
if "streetview_time_diff_days" in data.columns:
    data["photo_gap_months"] = pd.to_numeric(data["streetview_time_diff_days"], errors="coerce") / 30.44
    FAMILY["photo_gap_months"] = "Photo timing"
    PHOTO = ["photo_gap_months"]
    g = data["photo_gap_months"].iloc[:n]
    print(f"Photo timing: median {g.median():.1f} months; photo after the crash for {(g > 0).mean() * 100:.1f}% of crashes; "
          f"within 12 months for {(g.abs() <= 12).mean() * 100:.1f}%; missing {g.isna().mean() * 100:.1f}%.")
else:
    print("Photo timing column not found: the photo-timing version is skipped.")
print(f"Street feature candidates: {len(CANDIDATES)}; excluded: {[c for c in EXCLUDED_COLUMNS if c in CANDIDATES]}; "
      f"left: {len(BASE_ALL)}. Cyclist features (full model only): {CYCLIST}")

### 3.2 Missing values

Every empty value is filled, because models cannot work with empty cells.
A **marker** is an extra column holding 1 when the value was missing and
0 when it was there. The rule:

| Rule | Features | Why a value can be missing | Filled with | Marker |
|---|---|---|---|---|
| 1 | Points of interest | Nothing within reach | Median | Yes |
| 2 | Crash history counts | The history window starts before 2013, when the data starts | Median | Yes |
| 3 | Exposure, noise, deprivation, weather | Could not be matched at the location | Median, or "missing" for categories | Yes |
| 4 | Police-recorded fields, the cyclist's details, road feature pairs, earlier-severity ratios | Can depend on how the crash was recorded, and so on the injury | Median, or the most common category | No |

**A correction to the built features.** Road feature pairs join two
fields into one text (for example "Daylight" + "Single carriageway"). When
one field was missing, the first notebook wrote "nan" inside the text
(for example `Daylight__nan`), so the pair was never empty and rule 4 did
not apply. The same holds for the earlier-severity ratios, which were
calculated for the "missing" group. Missing police details are linked to
less serious crashes, probably because minor crashes are more often
reported with fewer details. So here a pair counts as missing when either
part is missing, and a ratio counts as missing when its source field is
missing. Both then follow rule 4.

In [ ]:
def marker_allowed(col):
    return FAMILY.get(col) in PLACE_FAMILIES

EMPTY_WORDS = {"nan", "none", "missing", "", "null"}
fix_rows = []
PAIR_COLS = [c for c in BASE_ALL if c.startswith("cx_")]
for c in PAIR_COLS:
    parts_missing = data[c].fillna("nan").astype(str).str.split("__").map(lambda parts: any(p.strip().lower() in EMPTY_WORDS for p in parts))
    newly = (parts_missing & data[c].notna()).iloc[:n]
    data.loc[parts_missing, c] = np.nan
    fix_rows.append({"feature": c, "kind": "road feature pair", "values set to missing": int(newly.sum()),
                     "share of crashes %": round(newly.mean() * 100, 2)})
RATIO_COLS = [c for c in BASE_ALL if c.startswith("prior_ratio_")]
for c in RATIO_COLS:
    source = c.replace("prior_ratio_", "", 1)
    if source not in data.columns:
        print(f"  {c}: source column '{source}' not found, left as it is")
        continue
    newly = (data[source].isna() & data[c].notna()).iloc[:n]
    data.loc[data[source].isna(), c] = np.nan
    fix_rows.append({"feature": c, "kind": "earlier-severity ratio", "values set to missing": int(newly.sum()),
                     "share of crashes %": round(newly.mean() * 100, 2)})
fix_table = pd.DataFrame(fix_rows).sort_values("values set to missing", ascending=False)
show(fix_table, "Values set to missing by the correction (all crashes from the start date):")
SHEETS["Missing correction"] = fix_table

fit_df = data.iloc[IDX["fit"]]
rows = []
for c in BASE_ALL + CYCLIST:
    miss = fit_df[c].isna()
    if miss.sum() >= 100:
        rows.append({"feature": c, "family": FAMILY.get(c), "missing (fitting part)": int(miss.sum()),
                     "serious % if missing": round(fit_df.loc[miss, "target"].mean() * 100, 1),
                     "serious % if present": round(fit_df.loc[~miss, "target"].mean() * 100, 1),
                     "marker": "yes" if marker_allowed(c) else "no (filled only)"})
missing_table = pd.DataFrame(rows).sort_values("missing (fitting part)", ascending=False)
by_family = missing_table.groupby(["family", "marker"]).size().rename("features").reset_index() if len(missing_table) else pd.DataFrame()
show(by_family, "Features with at least 100 missing values in the fitting part, by family:")
SHEETS["Missing values"] = missing_table
save_excel()

### 3.3 How strongly is each feature linked to severity?

Two measures, both on the fitting part only, after the missing-value rule:

- **Correlation.** For a number: the usual (Pearson) correlation with the
  outcome. For a category: each category with enough crashes becomes a
  yes/no column, and the strongest one counts.
- **Mutual information corrected for chance.** How much knowing the feature
  reduces uncertainty about the outcome, minus what the same calculation
  gives when the outcome is shuffled (chance), for every feature.

With n crashes, a correlation is different from zero at the usual
significance levels when it reaches z / sqrt(n): 1.96 / sqrt(n) for
p < 0.05, 2.576 / sqrt(n) for p < 0.01 and 3.291 / sqrt(n) for p < 0.001.
These are the correlation thresholds used for feature selection.

In [ ]:
STATS = {}

def association_stats(key, idx):
    """Correlation, chance-corrected mutual information and the inputs for mRMR, on the crashes in idx."""
    if key in STATS:
        return STATS[key]
    df = data.iloc[idx]
    yy = y_all[idx]
    min_cat = max(30, int(0.005 * len(idx)))
    X = pd.DataFrame(index=df.index)
    corr, best_cat, kind = {}, {}, {}
    for c in BASE_ALL:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            v = s.fillna(s.median()) if s.notna().any() else s.fillna(0)
            X[c] = v
            corr[c] = float(np.corrcoef(v, yy)[0, 1]) if v.std() > 0 else 0.0
            best_cat[c], kind[c] = "", "number"
        else:
            fill = "missing" if marker_allowed(c) else (s.mode().iloc[0] if s.notna().any() else "missing")
            t = s.fillna(fill).astype(str)
            freq = t.map(t.value_counts(normalize=True))
            X[c] = freq                                      # frequency code, used for redundancy only
            counts = t.value_counts()
            r_best, cat_best = 0.0, ""
            for cat in counts[counts >= min_cat].index:
                d = (t == cat).astype(float)
                if 0 < d.mean() < 1:
                    r = float(np.corrcoef(d, yy)[0, 1])
                    if abs(r) > abs(r_best):
                        r_best, cat_best = r, cat
            corr[c], best_cat[c], kind[c] = r_best, cat_best, "category"
    is_cat = [kind[c] == "category" for c in BASE_ALL]
    Xm = X.copy()
    for c, k in zip(BASE_ALL, is_cat):
        if k:
            fill = "missing" if marker_allowed(c) else (df[c].mode().iloc[0] if df[c].notna().any() else "missing")
            Xm[c] = OrdinalEncoder().fit_transform(df[[c]].fillna(fill).astype(str)).ravel()
    mi = mutual_info_classif(Xm.values, yy, discrete_features=is_cat, random_state=RANDOM_SEED)
    chance = np.mean([mutual_info_classif(Xm.values, np.random.RandomState(s).permutation(yy), discrete_features=is_cat,
                                          random_state=RANDOM_SEED) for s in range(CHANCE_SHUFFLES)], axis=0)
    out = pd.DataFrame({"feature": BASE_ALL, "type": [kind[c] for c in BASE_ALL], "family": [FAMILY.get(c) for c in BASE_ALL],
                        "strongest category": [best_cat[c] for c in BASE_ALL], "correlation": [corr[c] for c in BASE_ALL],
                        "mutual information": mi, "chance level": chance})
    out["|correlation|"] = out["correlation"].abs()
    out["relevance (MI minus chance)"] = (out["mutual information"] - out["chance level"]).clip(lower=0)
    Z = X.values.astype(float)
    Z = (Z - Z.mean(axis=0)) / np.where(Z.std(axis=0) > 0, Z.std(axis=0), 1)
    R = np.abs(np.nan_to_num(np.corrcoef(Z, rowvar=False)))
    STATS[key] = {"table": out.set_index("feature"), "R": R, "n": len(idx)}
    return STATS[key]


def correlations_only(idx):
    """|correlation| of every street feature with severity on the crashes in idx (same rule as above)."""
    df, yy = data.iloc[idx], y_all[idx]
    min_cat = max(30, int(0.005 * len(idx)))
    out = {}
    for c in BASE_ALL:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            v = s.fillna(s.median()) if s.notna().any() else s.fillna(0)
            out[c] = abs(float(np.corrcoef(v, yy)[0, 1])) if v.std() > 0 else 0.0
        else:
            fill = "missing" if marker_allowed(c) else (s.mode().iloc[0] if s.notna().any() else "missing")
            t = s.fillna(fill).astype(str)
            counts = t.value_counts()
            best = 0.0
            for cat in counts[counts >= min_cat].index:
                d = (t == cat).astype(float)
                if 0 < d.mean() < 1:
                    best = max(best, abs(float(np.corrcoef(d, yy)[0, 1])))
            out[c] = best
    return pd.Series(out)


SELECTED = {}

def select_features(option, key, idx, fixed_k=None):
    """The street features kept by one selection option, using only the crashes in idx.
    fixed_k: when the model is trained again on more data, keep the number of features chosen before."""
    if (option, key, fixed_k) in SELECTED:
        return SELECTED[(option, key, fixed_k)]
    st = association_stats(key, idx)
    tab = st["table"]
    if option == "All features":
        cols = list(BASE_ALL)
    elif option == STABLE_OPTION:
        rng = np.random.RandomState(RANDOM_SEED)
        cut = P_VALUE_LEVELS["Correlation p<0.01"] / np.sqrt(len(idx))
        votes = pd.Series(0, index=BASE_ALL)
        for _ in range(5):
            sub = np.sort(rng.choice(idx, size=len(idx) // 2, replace=False))
            votes += (correlations_only(sub) >= cut).astype(int)
        ranked = pd.DataFrame({"votes": votes, "r": tab["|correlation|"]}).sort_values(["votes", "r"], ascending=False)
        cols = ranked.index[:fixed_k].tolist() if fixed_k else ranked.index[ranked["votes"] >= 3].tolist()
        if len(cols) < MIN_FEATURES:
            cols = ranked.index[:MIN_FEATURES].tolist()
    elif fixed_k and (option in P_VALUE_LEVELS or option in FIXED_CORRELATION):
        cols = tab["|correlation|"].sort_values(ascending=False).index[:fixed_k].tolist()
    elif option in P_VALUE_LEVELS or option in FIXED_CORRELATION:
        cut = P_VALUE_LEVELS[option] / np.sqrt(st["n"]) if option in P_VALUE_LEVELS else FIXED_CORRELATION[option]
        cols = tab.index[tab["|correlation|"] >= cut].tolist()
        if len(cols) < MIN_FEATURES:
            cols = tab["|correlation|"].sort_values(ascending=False).index[:MIN_FEATURES].tolist()
    else:
        k = int(option.split()[-1])
        rel = tab["relevance (MI minus chance)"].values
        order = list(np.argsort(-rel))
        chosen = [order[0]]
        remaining = [i for i in order[1:]]
        while len(chosen) < min(k, len(BASE_ALL)) and remaining:
            red = st["R"][np.ix_(remaining, chosen)].mean(axis=1)
            score = rel[remaining] / (red + 0.01)
            if score.max() <= 0:
                extra = tab["|correlation|"].values
                pick = remaining[int(np.argmax(extra[remaining]))]
            else:
                pick = remaining[int(np.argmax(score))]
            chosen.append(pick)
            remaining.remove(pick)
        cols = [BASE_ALL[i] for i in chosen]
    SELECTED[(option, key, fixed_k)] = cols
    return cols


t0 = time.time()
fit_stats = association_stats("fit", IDX["fit"])
assoc = fit_stats["table"].reset_index().sort_values("|correlation|", ascending=False)
n_fit = len(IDX["fit"])
print(f"Association measured on {n_fit:,} fitting crashes in {time.time() - t0:.0f} s.")
summary = assoc["|correlation|"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])
show(summary.rename({"count": "features", "mean": "average", "std": "spread", "min": "lowest", "50%": "median", "max": "highest"})
     .to_frame("|correlation|"), "How strong are the correlations?")
print(f"Rank agreement between correlation and mutual information: "
      f"{assoc['|correlation|'].rank().corr(assoc['relevance (MI minus chance)'].rank()):.3f}")
option_rows = []
for option in OPTIONS:
    cols = select_features(option, "fit", IDX["fit"])
    rule = (f"|r| >= {P_VALUE_LEVELS[option]} / sqrt({n_fit:,}) = {P_VALUE_LEVELS[option] / np.sqrt(n_fit):.4f}" if option in P_VALUE_LEVELS
            else f"|r| >= {FIXED_CORRELATION[option]}" if option in FIXED_CORRELATION
            else "p<0.01 in at least 3 of 5 random halves" if option == STABLE_OPTION
            else "most relevant, least redundant" if option.startswith("mRMR") else "no filter")
    option_rows.append({"Selection option": option, "Rule": rule, "Street features kept": len(cols),
                        "Numbers": sum(fit_stats["table"].at[c, "type"] == "number" for c in cols),
                        "Categories": sum(fit_stats["table"].at[c, "type"] == "category" for c in cols)})
options_table = pd.DataFrame(option_rows)
show(options_table, "\nThe 10 feature-selection options, applied to the fitting part:")
show(assoc.head(20)[["feature", "family", "type", "strongest category", "correlation", "relevance (MI minus chance)"]],
     "\n20 features with the strongest correlation:")
show(assoc.sort_values("relevance (MI minus chance)", ascending=False).head(20)[
    ["feature", "family", "type", "|correlation|", "relevance (MI minus chance)"]], "\n20 features with the strongest mutual information (minus chance):")
SHEETS["Feature association"] = assoc.round(6)
SHEETS["Selection options"] = options_table

fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 2.8))
axes[0].hist(assoc["|correlation|"], bins=40, color=BLUE)
for option, z in P_VALUE_LEVELS.items():
    axes[0].axvline(z / np.sqrt(n_fit), color=RED, linestyle="--", linewidth=0.8)
axes[0].axvline(0.03, color=ORANGE, linestyle="--", linewidth=0.8)
axes[0].set_xlabel("|correlation| with severity")
axes[0].set_ylabel("Features")
axes[0].set_title(f"Correlation of the {len(BASE_ALL)} features (dashed = thresholds)")
axes[1].barh(options_table["Selection option"][::-1], options_table["Street features kept"][::-1], color=BLUE)
axes[1].set_xlabel("Street features kept")
axes[1].set_title("Features kept by each option")
plt.tight_layout()
save_chart(fig, "02_feature_association_and_options")
save_excel()

## Section 4: model search

**How every setting is scored.** Five time-ordered folds inside the
fitting part: each fold trains on older crashes and checks on the period
just after them. The score is the average ROC-AUC over the five folds.

**What a setting contains.** The model's own settings, plus one of the 10
feature-selection options. Inside each fold, the features are selected
again using only that fold's training crashes, so the checking crashes
never influence the selection.

The search uses the full model's features (selected street features +
age, sex and age group). The validation and test parts play no role.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from joblib import Parallel, delayed

SKLEARN_NEW = tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 8)

def gpu_ok_xgb():
    try:
        xgb.XGBClassifier(n_estimators=2, tree_method="hist", device="cuda").fit(np.random.rand(100, 3), np.random.randint(0, 2, 100))
        return True
    except Exception:
        return False

XGB_DEVICE = "cuda" if gpu_ok_xgb() else "cpu"
INPUT_KIND = {"XGBoost": "tree", "LightGBM": "tree", "Elastic-net logistic regression": "linear"}
print(f"XGBoost {xgb.__version__} on {XGB_DEVICE}; LightGBM {lgb.__version__}; scikit-learn {sklearn.__version__}")


def build_preprocessor(df, cols, scale):
    cats = [c for c in cols if not pd.api.types.is_numeric_dtype(df[c])]
    nums = [c for c in cols if c not in cats]
    scaler = [("scale", StandardScaler())] if scale else []
    onehot = ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, max_categories=50))
    groups = {
        "num_marked": ([c for c in nums if marker_allowed(c)], [("impute", SimpleImputer(strategy="median", add_indicator=True))] + scaler),
        "num_plain": ([c for c in nums if not marker_allowed(c)], [("impute", SimpleImputer(strategy="median"))] + scaler),
        "cat_marked": ([c for c in cats if marker_allowed(c)], [("impute", SimpleImputer(strategy="constant", fill_value="missing")), onehot]),
        "cat_plain": ([c for c in cats if not marker_allowed(c)], [("impute", SimpleImputer(strategy="most_frequent")), onehot]),
    }
    return ColumnTransformer([(k, Pipeline(steps), g) for k, (g, steps) in groups.items() if g])


def prepare(kind, train_idx, eval_idx, cols):
    train, other = data.iloc[train_idx], data.iloc[eval_idx]
    pre = build_preprocessor(train, cols, scale=(kind == "linear"))
    X_tr, X_ev = pre.fit_transform(train[cols]), pre.transform(other[cols])
    return X_tr, X_ev, list(pre.get_feature_names_out())


def sample_weights(y, base=None):
    """Balanced weights: serious and slight crashes weigh the same in total (base = weights before balancing)."""
    w = np.ones(len(y)) if base is None else np.asarray(base, dtype=float).copy()
    pos, neg = w[y == 1].sum(), w[y == 0].sum()
    w[y == 1] *= neg / max(pos, 1e-9)
    return w


def make_model(model_type, params, seed):
    p = {k: v for k, v in params.items() if k != "selection"}
    if model_type == "XGBoost":
        return xgb.XGBClassifier(**p, tree_method="hist", device=XGB_DEVICE, random_state=seed, eval_metric="logloss", n_jobs=N_CPU)
    if model_type == "LightGBM":
        return lgb.LGBMClassifier(**p, subsample_freq=1, random_state=seed, verbosity=-1, n_jobs=N_CPU)
    s = dict(C=p["C"], l1_ratio=p["l1_ratio"], solver="saga", max_iter=2000, tol=1e-3, random_state=seed)
    if not SKLEARN_NEW:
        s["penalty"] = "elasticnet"
    return LogisticRegression(**s)


def fit_predict(model_type, params, X_tr, y, X_ev, seed=RANDOM_SEED, base_weights=None):
    m = make_model(model_type, params, seed)
    m.fit(X_tr, y, sample_weight=sample_weights(y, base_weights))
    return m.predict_proba(X_ev)[:, 1], m


def fast_auc(y, p):
    r = rankdata(p)
    n1 = y.sum()
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(y) - n1))


FIT = IDX["fit"]
edges = np.linspace(0, len(FIT), N_FOLDS + 2).astype(int)
FOLDS = [(FIT[:edges[i]], FIT[edges[i]:edges[i + 1]]) for i in range(1, N_FOLDS + 1)]
fold_table = pd.DataFrame([{"Fold": i + 1, "Train crashes": len(tr), "Check crashes": len(te),
                            "Check period": f"{data.iloc[te]['accident_dt'].min():%Y-%m} to {data.iloc[te]['accident_dt'].max():%Y-%m}"}
                           for i, (tr, te) in enumerate(FOLDS)])
show(fold_table, "The five time-ordered folds:")

t0 = time.time()
for k, (tr, te) in enumerate(FOLDS):
    for option in OPTIONS:
        select_features(option, f"fold{k}", tr)
print(f"Feature selection done for every fold and option in {(time.time() - t0) / 60:.1f} min (training crashes of each fold only).")
FOLD_X = {}

def fold_inputs(k, option, kind):
    key = (k, option, kind)
    if key not in FOLD_X:
        tr, te = FOLDS[k]
        cols = SELECTED[(option, f"fold{k}", None)] + CYCLIST
        X_tr, X_te, _ = prepare(kind, tr, te, cols)
        FOLD_X[key] = (X_tr, X_te)
    return FOLD_X[key]


def _score(model_type, params, X_tr, y_tr, X_te, y_te):
    p, _ = fit_predict(model_type, params, X_tr, y_tr, X_te)
    return fast_auc(y_te, p)


def cv_score(model_type, params):
    kind = INPUT_KIND[model_type]
    jobs = []
    for k, (tr, te) in enumerate(FOLDS):
        X_tr, X_te = fold_inputs(k, params["selection"], kind)
        jobs.append((model_type, params, X_tr, y_all[tr], X_te, y_all[te]))
    if model_type == "Elastic-net logistic regression":
        aucs = Parallel(n_jobs=min(N_CPU, N_FOLDS))(delayed(_score)(*j) for j in jobs)
    else:
        aucs = [_score(*j) for j in jobs]
    return float(np.mean(aucs)), float(np.std(aucs)), aucs


def plain(params):
    out = {}
    for k, v in params.items():
        if isinstance(v, (float, np.floating)):
            out[k] = round(float(v), 6)
        elif isinstance(v, (int, np.integer)):
            out[k] = int(v)
        else:
            out[k] = v
    return out

SPACES = {
    "XGBoost": {"n_estimators": randint(100, 901), "max_depth": randint(1, 9), "learning_rate": loguniform(0.01, 0.3),
                "subsample": uniform(0.5, 0.5), "colsample_bytree": uniform(0.3, 0.7), "min_child_weight": randint(1, 51),
                "reg_lambda": loguniform(0.1, 50), "reg_alpha": uniform(0, 2), "selection": OPTIONS},
    "Elastic-net logistic regression": {"C": loguniform(1e-4, 1), "l1_ratio": uniform(0.05, 0.95), "selection": OPTIONS},
    "LightGBM": {"n_estimators": randint(100, 901), "max_depth": randint(1, 9), "num_leaves": randint(3, 64),
                 "learning_rate": loguniform(0.01, 0.3), "subsample": uniform(0.5, 0.5), "colsample_bytree": uniform(0.3, 0.7),
                 "min_child_samples": randint(20, 301), "reg_lambda": loguniform(0.1, 50), "selection": OPTIONS},
}
SEARCH = []
print("Search tools ready.")

### 4.1 The search: 100 settings for each model type

| Model type | Settings drawn at random from |
|---|---|
| XGBoost | trees 100 to 900; depth 1 to 8; learning rate 0.01 to 0.3 (log scale); rows per tree 0.5 to 1; columns per tree 0.3 to 1; minimum child weight 1 to 50; L2 penalty 0.1 to 50 (log); L1 penalty 0 to 2 |
| Elastic-net logistic regression | C 0.0001 to 1 (log scale); l1_ratio 0.05 to 1 |
| LightGBM | trees 100 to 900; depth 1 to 8; leaves 3 to 63; learning rate 0.01 to 0.3 (log); rows 0.5 to 1; columns 0.3 to 1; minimum leaf size 20 to 300; L2 penalty 0.1 to 50 (log) |

Every setting also draws one of the 10 feature-selection options.

In [ ]:
for model_type in MODEL_TYPES:
    t0 = time.time()
    for i, params in enumerate(ParameterSampler(SPACES[model_type], n_iter=N_SETTINGS, random_state=RANDOM_SEED)):
        s0 = time.time()
        mean, std, aucs = cv_score(model_type, params)
        SEARCH.append({"Model type": model_type, "Setting": i + 1, "Selection": params["selection"],
                       "Settings": json.dumps({k: v for k, v in plain(params).items() if k != "selection"}),
                       "Mean ROC-AUC (5 folds)": mean, "Spread": std, **{f"Fold {k + 1}": a for k, a in enumerate(aucs)},
                       "Seconds": round(time.time() - s0, 1), "_params": params})
        if (i + 1) % 25 == 0:
            best = max(r["Mean ROC-AUC (5 folds)"] for r in SEARCH if r["Model type"] == model_type)
            print(f"  {model_type}: {i + 1} settings, best so far {best:.4f}, {(time.time() - t0) / 60:.1f} min")
    print(f"{model_type}: done in {(time.time() - t0) / 60:.1f} min")
    SHEETS["Search"] = pd.DataFrame(SEARCH).drop(columns="_params").round(6)
    save_excel()
search_all = pd.DataFrame(SEARCH)
del FOLD_X

### 4.2 Search results

In [ ]:
for model_type in MODEL_TYPES:
    top = search_all[search_all["Model type"] == model_type].sort_values("Mean ROC-AUC (5 folds)", ascending=False).head(10)
    show(top[["Setting", "Selection", "Settings", "Mean ROC-AUC (5 folds)", "Spread"]].reset_index(drop=True),
         f"\n{model_type}: ten best settings")

by_option = search_all.pivot_table(index="Selection", columns="Model type", values="Mean ROC-AUC (5 folds)", aggfunc="max").reindex(OPTIONS)[MODEL_TYPES]
counts_option = search_all.pivot_table(index="Selection", columns="Model type", values="Setting", aggfunc="count").reindex(OPTIONS)[MODEL_TYPES]
by_option_view = by_option.copy()
by_option_view.columns = [f"{c}: best ROC-AUC" for c in by_option.columns]
for c in MODEL_TYPES:
    by_option_view[f"{c}: settings"] = counts_option[c].fillna(0).astype(int)
by_option_view.insert(0, "Street features (fitting part)", options_table.set_index("Selection option")["Street features kept"].reindex(OPTIONS).values)
show(by_option_view.reset_index(), "\nBest score of each feature-selection option, per model type:")
SHEETS["Search by selection"] = by_option_view.reset_index().round(6)

best_rows = search_all.loc[search_all.groupby("Model type")["Mean ROC-AUC (5 folds)"].idxmax()].set_index("Model type").reindex(MODEL_TYPES)
BEST = {m: best_rows.at[m, "_params"] for m in MODEL_TYPES}
show(best_rows.reset_index()[["Model type", "Selection", "Settings", "Mean ROC-AUC (5 folds)", "Spread"]],
     "\nBest setting of each model type:")
SHEETS["Search best per type"] = best_rows.reset_index().drop(columns="_params").round(6)
save_excel()

colours = {"XGBoost": BLUE, "Elastic-net logistic regression": ORANGE, "LightGBM": "#3A9E5F"}
fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 3.2), gridspec_kw={"width_ratios": [1, 1.4]})
for k, m in enumerate(MODEL_TYPES):
    sc = search_all.loc[search_all["Model type"] == m, "Mean ROC-AUC (5 folds)"].values
    axes[0].scatter(np.full(len(sc), k) + np.random.RandomState(k).uniform(-0.22, 0.22, len(sc)), sc, s=7, alpha=0.6, color=colours[m])
    axes[0].scatter(k, sc.max(), s=45, marker="D", color=colours[m], edgecolor="black", linewidth=0.6, zorder=3)
axes[0].set_xticks(range(len(MODEL_TYPES)))
axes[0].set_xticklabels(MODEL_TYPES)
axes[0].set_ylabel("Mean ROC-AUC over 5 folds")
axes[0].set_title("Every setting (diamond = best)")
yy = np.arange(len(OPTIONS))
for k, m in enumerate(MODEL_TYPES):
    axes[1].scatter(by_option[m].values, yy + (k - 1) * 0.22, color=colours[m], s=18, label=m)
axes[1].set_yticks(yy)
axes[1].set_yticklabels(OPTIONS)
axes[1].invert_yaxis()
axes[1].set_xlabel("Best mean ROC-AUC")
axes[1].set_title("Best score by feature-selection option")
axes[1].legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
plt.tight_layout()
save_chart(fig, "03_search_overview")

## Section 5: the best model

**The rule, fixed in advance:** there are four candidates: the best
setting of each model type, and the average of those three. Each is
trained on the whole fitting part and scored on the validation part. The
candidate with the **highest balanced accuracy on the validation part** is
the final model (ROC-AUC breaks a tie).

Balanced accuracy on the validation part is measured so that the cut-off
is never chosen on the crashes it is graded on: the cut-off with the best
balanced accuracy is chosen on the first half of the validation part, and
the result is read on the second half.

In [ ]:
def best_cut(y, p):
    cuts = np.unique(np.quantile(p, np.linspace(0.01, 0.99, 197)))
    return cuts[int(np.argmax([((p >= c)[y == 1].mean() + (p < c)[y == 0].mean()) / 2 for c in cuts]))]

def at_cut(y, p, cut):
    flag = p >= cut
    recall, spec = flag[y == 1].mean(), (~flag)[y == 0].mean()
    return {"balanced_accuracy": (recall + spec) / 2, "recall": recall, "specificity": spec,
            "precision": y[flag].mean() if flag.any() else 0.0, "flagged": flag.mean()}

def honest_val_ba(p):
    cut = best_cut(y_all[IDX["val1"]], p[:half])
    return at_cut(y_all[IDX["val2"]], p[half:], cut)["balanced_accuracy"]

RESULTS = {}

USE_OLDER = False      # decided in Section 5.3

def training_rows(train_key, use_older):
    """Training rows, labels and weights. With older crashes: each appears twice, as serious with weight p
    and as slight with weight 1 - p (rows with weight 0 are dropped)."""
    tr = IDX[train_key]
    if not (use_older and len(IDX["older"])):
        return tr, y_all[tr], np.ones(len(tr))
    p = OLDER_P
    pos, neg = p > 0, p < 1
    rows = np.concatenate([tr, IDX["older"][pos], IDX["older"][neg]])
    labels = np.concatenate([y_all[tr], np.ones(pos.sum(), int), np.zeros(neg.sum(), int)])
    weights = np.concatenate([np.ones(len(tr)), p[pos], 1 - p[neg]])
    return rows, labels, weights


def train_one(model_type, params, cols, train_key, eval_key, keep_model=False, use_older=None):
    use_older = USE_OLDER if use_older is None else use_older
    tr, y_tr, w_tr = training_rows(train_key, use_older)
    ev = IDX[eval_key]
    X_tr, X_ev, names = prepare(INPUT_KIND[model_type], tr, ev, cols)
    seeds = [RANDOM_SEED] if model_type == "Elastic-net logistic regression" else SEEDS
    preds, models = [], []
    for s in seeds:
        p, m = fit_predict(model_type, params, X_tr, y_tr, X_ev, s, base_weights=w_tr)
        preds.append(p)
        models.append(m)
    out = {"pred": pd.Series(np.mean(preds, axis=0), index=ev), "cols": cols}
    if keep_model:
        pre = build_preprocessor(data.iloc[tr], cols, scale=(INPUT_KIND[model_type] == "linear")).fit(data.iloc[tr][cols])
        out.update({"models": models, "X_eval": X_ev, "X_train": X_tr, "names": names, "preprocessor": pre})
    return out

t0 = time.time()
choice_rows, VAL_PRED = [], {}
for m in MODEL_TYPES:
    option = BEST[m]["selection"]
    cols = select_features(option, "fit", IDX["fit"]) + CYCLIST
    r = train_one(m, BEST[m], cols, "fit", "val")
    p = r["pred"].loc[IDX["val"]].values
    VAL_PRED[m] = p
    choice_rows.append({"Model type": m, "Selection": option, "Features": len(cols),
                        "Mean ROC-AUC (5 folds)": best_rows.at[m, "Mean ROC-AUC (5 folds)"],
                        "Validation ROC-AUC": fast_auc(yv, p), "Validation balanced accuracy %": honest_val_ba(p) * 100})
p_avg = np.mean([VAL_PRED[m] for m in MODEL_TYPES], axis=0)
choice_rows.append({"Model type": AVERAGE, "Selection": "each model its own", "Features": np.nan, "Mean ROC-AUC (5 folds)": np.nan,
                    "Validation ROC-AUC": fast_auc(yv, p_avg), "Validation balanced accuracy %": honest_val_ba(p_avg) * 100})
choice = pd.DataFrame(choice_rows).sort_values(["Validation balanced accuracy %", "Validation ROC-AUC"], ascending=False).reset_index(drop=True)
FINAL_TYPE = choice.at[0, "Model type"]
MEMBERS = list(MODEL_TYPES) if FINAL_TYPE == AVERAGE else [FINAL_TYPE]
FINAL_OPTION = " / ".join(dict.fromkeys(BEST[m]["selection"] for m in MEMBERS))
show(choice, "The four candidates on the validation part (full model):")
print(f"\nFinal model: {FINAL_TYPE}")
for m in MEMBERS:
    print(f"  {m}: selection '{BEST[m]['selection']}', settings {json.dumps({k: v for k, v in plain(BEST[m]).items() if k != 'selection'})}")
print(f"Time: {(time.time() - t0) / 60:.1f} min")
SHEETS["Model choice"] = choice.round(6)
save_excel()

### 5.2 Do the older crashes help?

Crashes before November 2016 use the old severity recording, so they are
never used for validation or test. They can still help the model learn,
if their labels are put on the new scale. The Department for Transport
gives, for each casualty, the probability that the injury would have been
recorded as serious under injury-based recording. Each older crash then
enters training twice: as serious with weight p, and as slight with
weight 1 - p.

**The rule, fixed in advance:** the final model is trained on the fitting
part with and without the older crashes, and scored on the validation
part. The older crashes are used from here on only if they give a higher
validation balanced accuracy and do not lower the ROC-AUC. The model's
settings and features stay the same.

In [ ]:
t0 = time.time()
older_rows = []
p_without = np.mean([VAL_PRED[m] for m in MEMBERS], axis=0)
older_rows.append({"Training": "Fitting part only (from November 2016)", "Training crashes": len(IDX["fit"]),
                   "Validation ROC-AUC": fast_auc(yv, p_without), "Validation balanced accuracy %": honest_val_ba(p_without) * 100})
if OLDER_OK:
    p_with = np.mean([train_one(m, BEST[m], select_features(BEST[m]["selection"], "fit", IDX["fit"]) + CYCLIST, "fit", "val",
                                use_older=True)["pred"].loc[IDX["val"]].values for m in MEMBERS], axis=0)
    older_rows.append({"Training": "Fitting part + older crashes (adjusted labels)", "Training crashes": len(IDX["fit"]) + len(IDX["older"]),
                       "Validation ROC-AUC": fast_auc(yv, p_with), "Validation balanced accuracy %": honest_val_ba(p_with) * 100})
older_table = pd.DataFrame(older_rows)
if OLDER_OK:
    gain = older_table.at[1, "Validation balanced accuracy %"] - older_table.at[0, "Validation balanced accuracy %"]
    auc_change = older_table.at[1, "Validation ROC-AUC"] - older_table.at[0, "Validation ROC-AUC"]
    USE_OLDER = bool(gain > OLDER_MIN_GAIN and auc_change >= 0)
    print(f"Change with the older crashes: balanced accuracy {gain:+.2f} points, ROC-AUC {auc_change:+.4f}.")
show(older_table, "Final model on the validation part, with and without the older crashes:")
print(f"Decision: older crashes {'ARE' if USE_OLDER else 'are NOT'} used for training from here on.")
SHEETS["Older crashes"] = older_table.round(5)
save_excel()
print(f"Time: {(time.time() - t0) / 60:.1f} min")

### 5.3 Every version on the validation part

The final model is trained on the fitting part for every version. Street
features are selected with the final model's selection option on the
fitting part; the full model adds age, sex and age group; the safety
versions add their columns on top.

| Version | Columns added to the base |
|---|---|
| + safety, all six | the four photo scores, their average and their spread |
| + safety, mean | the average (**main question**, on the full model) |
| + safety, mean and std | the average and the spread |
| + safety, worst case | the lowest of the four |
| Safety score only | only the perceived safety columns (a reference) |
| Cyclist only | only age, sex and age group (a reference) |
| Full + photo timing | the full model plus the months between the crash and the street photo |

In [ ]:
HEADINGS = [c for c in SAFETY_COLS if c.startswith("safety_score_heading_")]
SAFETY_VERSIONS = {"all six": HEADINGS + ["safety_score_mean", "safety_score_std"], "mean": ["safety_score_mean"],
                   "mean and std": ["safety_score_mean", "safety_score_std"], "worst case": ["safety_score_min"]}
SAFETY_ONLY_COLS = HEADINGS + ["safety_score_mean", "safety_score_std", "safety_score_min"]

def make_versions(street_cols):
    """The versions: street and full, each without and with the safety score, two references and photo timing."""
    out = {}
    for b, cols in {"Street": street_cols, "Full": street_cols + CYCLIST}.items():
        out[b] = cols
        for v, add in SAFETY_VERSIONS.items():
            out[f"{b} + safety {v}"] = cols + add
    out["Safety score only"] = SAFETY_ONLY_COLS
    out["Cyclist only"] = list(CYCLIST)
    if PHOTO:
        out["Full + photo timing"] = street_cols + CYCLIST + PHOTO
    return out

def street_for(m, key, idx, fixed_k=None):
    return select_features(BEST[m]["selection"], key, idx, fixed_k)

def train_versions(train_key, eval_key, select_key, select_idx, fixed=None, only=None, suffix="", keep=("Full", "Street")):
    """Trains every member of the final model for each version. Each member uses its own feature selection."""
    t0 = time.time()
    for m in MEMBERS:
        street = street_for(m, select_key, select_idx, None if fixed is None else fixed[m])
        versions = make_versions(street)
        for name in (only or list(versions)):
            key = (m, name + suffix, train_key)
            if key not in RESULTS:
                RESULTS[key] = train_one(m, BEST[m], versions[name], train_key, eval_key, keep_model=name in keep)
    print(f"  {len(MEMBERS)} model(s) x {len(only or VERSIONS)} versions trained on '{train_key}' in {(time.time() - t0) / 60:.1f} min")

def pred(name, train_key, part):
    """The final model's scores: one model, or the average of the three."""
    return np.mean([RESULTS[(m, name, train_key)]["pred"].loc[IDX[part]].values for m in MEMBERS], axis=0)

STREET_FIT = sorted(set().union(*[street_for(m, "fit", IDX["fit"]) for m in MEMBERS]))
FIXED_K = {m: len(street_for(m, "fit", IDX["fit"])) for m in MEMBERS}
VERSIONS = make_versions(street_for(MEMBERS[0], "fit", IDX["fit"]))
train_versions("fit", "val+test", "fit", IDX["fit"])
val_rows = [{"Version": name, "Features (first model)": len(RESULTS[(MEMBERS[0], name, "fit")]["cols"]),
             "Validation ROC-AUC": fast_auc(yv, pred(name, "fit", "val")),
             "Validation balanced accuracy %": honest_val_ba(pred(name, "fit", "val")) * 100} for name in VERSIONS]
val_table = pd.DataFrame(val_rows)
show(val_table, f"Every version on the validation part ({FINAL_TYPE}):", 3)
SHEETS["Validation results"] = val_table.round(5)
save_excel()

## Section 6: test results

**Main result.** The whole method is repeated on all crashes before the
test period (fitting + validation), with **exactly the same street
features as the validated model** and the same settings. Only the
training data grows, so the model tested is the model that was validated. A final model is normally trained on all
data available before it is used.

**Cut-off.** The cut-off is the point above which a crash is flagged as
serious, and it is set to give the best balanced accuracy. The
validation part is now in the training data, so the cut-off is found
like this: the same version trained on the fitting part only is scored
on the validation part, the cut-off with the best balanced accuracy is
found there, and the main model flags the same share of test crashes.
No test outcome is used.

Every number has a 95% range from 500 resamples of the test crashes.

In [ ]:
t0 = time.time()
STREET_MAIN = STREET_FIT
train_versions("fit+val", "test", "fit", IDX["fit"])
for m in MEMBERS:
    print(f"{m}: the same {len(street_for(m, 'fit', IDX['fit']))} street features as the validated model.")

CUTS, SHARES = {}, {}
for name in VERSIONS:
    p_val = pred(name, "fit", "val")
    c = best_cut(yv, p_val)
    SHARES[name] = (p_val >= c).mean()
    CUTS[name] = np.quantile(pred(name, "fit+val", "test"), 1 - SHARES[name])

def ranges(y, p, cut, n_resamples=BOOTSTRAP_RANGES):
    rng = np.random.RandomState(RANDOM_SEED)
    draws = []
    for _ in range(n_resamples):
        i = rng.randint(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        r = at_cut(y[i], p[i], cut)
        r["roc_auc"] = fast_auc(y[i], p[i])
        draws.append(r)
    draws = pd.DataFrame(draws)
    return {k: np.percentile(draws[k], [2.5, 97.5]) for k in draws.columns}

rows = []
for name in VERSIONS:
    p = pred(name, "fit+val", "test")
    m, r = at_cut(yt, p, CUTS[name]), ranges(yt, p, CUTS[name])
    rows.append({"Version": name, "Balanced accuracy %": m["balanced_accuracy"] * 100,
                 "BA 95% range": f"{r['balanced_accuracy'][0] * 100:.1f} to {r['balanced_accuracy'][1] * 100:.1f}",
                 "BA low": r["balanced_accuracy"][0] * 100, "BA high": r["balanced_accuracy"][1] * 100,
                 "Recall %": m["recall"] * 100, "Specificity %": m["specificity"] * 100, "Precision %": m["precision"] * 100,
                 "ROC-AUC": fast_auc(yt, p), "ROC-AUC 95% range": f"{r['roc_auc'][0]:.3f} to {r['roc_auc'][1]:.3f}",
                 "PR-AUC": average_precision_score(yt, p), "Flagged %": m["flagged"] * 100, "Cut-off": CUTS[name]})
test_table = pd.DataFrame(rows)
print(f"Test part: {len(yt):,} crashes, {yt.mean() * 100:.1f}% serious or fatal. Final model: {FINAL_TYPE}, selection '{FINAL_OPTION}'.")
show(test_table.drop(columns=["BA low", "BA high", "Cut-off"]), digits=3)
SHEETS["Test results"] = test_table.round(5)

def paired_ba_difference(name_ref, name_new, n_resamples=BOOTSTRAP_DIFFERENCES):
    p0, p1 = pred(name_ref, "fit+val", "test"), pred(name_new, "fit+val", "test")
    c0, c1 = CUTS[name_ref], CUTS[name_new]
    rng = np.random.RandomState(RANDOM_SEED)
    d = []
    for _ in range(n_resamples):
        i = rng.randint(0, len(yt), len(yt))
        d.append(at_cut(yt[i], p1[i], c1)["balanced_accuracy"] - at_cut(yt[i], p0[i], c0)["balanced_accuracy"])
    diff = at_cut(yt, p1, c1)["balanced_accuracy"] - at_cut(yt, p0, c0)["balanced_accuracy"]
    return diff * 100, np.percentile(d, 2.5) * 100, np.percentile(d, 97.5) * 100

comp_rows = []
tt = test_table.set_index("Version")
for b in ["Street", "Full"]:
    for v in SAFETY_VERSIONS:
        name = f"{b} + safety {v}"
        d, lo, hi = paired_ba_difference(b, name)
        comp_rows.append({"Comparison": f"{name} vs {b}", "BA without safety %": tt.at[b, "Balanced accuracy %"],
                          "BA with safety %": tt.at[name, "Balanced accuracy %"], "Difference (points)": d,
                          "Difference 95% range": f"{lo:+.2f} to {hi:+.2f}", "Low": lo, "High": hi,
                          "Main question": (b == "Full" and v == "mean")})
if PHOTO:
    d, lo, hi = paired_ba_difference("Full", "Full + photo timing")
    comp_rows.append({"Comparison": "Full + photo timing vs Full", "BA without safety %": tt.at["Full", "Balanced accuracy %"],
                      "BA with safety %": tt.at["Full + photo timing", "Balanced accuracy %"], "Difference (points)": d,
                      "Difference 95% range": f"{lo:+.2f} to {hi:+.2f}", "Low": lo, "High": hi, "Main question": False})
comparisons = pd.DataFrame(comp_rows).rename(columns={"BA without safety %": "BA without %", "BA with safety %": "BA with %"})
show(comparisons.drop(columns=["Low", "High"]), "\nBalanced accuracy with and without the added columns (main question marked):", 2)
SHEETS["Test comparisons"] = comparisons.round(4)
save_excel()
print(f"Time: {(time.time() - t0) / 60:.1f} min")

### 6.2 The models for the paper: the final model and the three single models

The final model is the one chosen by the rule fixed in advance. The single
models are shown next to it to show that the conclusions do not depend on
the model type.

In [ ]:
# The single models that are not part of the final model are trained here too (full model only), so the table
# shows every model on validation and test. This does not change any other result.
for m in MODEL_TYPES:
    cols_m = street_for(m, "fit", IDX["fit"]) + CYCLIST
    for train_key, eval_key in [("fit", "val+test"), ("fit+val", "test")]:
        if (m, "Full", train_key) not in RESULTS:
            RESULTS[(m, "Full", train_key)] = train_one(m, BEST[m], cols_m, train_key, eval_key)
paper_rows = []
for m in MODEL_TYPES + [AVERAGE]:
    members = MODEL_TYPES if m == AVERAGE else [m]
    pv = np.mean([RESULTS[(x, "Full", "fit")]["pred"].loc[IDX["val"]].values for x in members], axis=0)
    ptst = np.mean([RESULTS[(x, "Full", "fit+val")]["pred"].loc[IDX["test"]].values for x in members], axis=0)
    cut = np.quantile(ptst, 1 - (pv >= best_cut(yv, pv)).mean())
    r = ranges(yt, ptst, cut)
    paper_rows.append({"Model": m + (" (final model)" if m == FINAL_TYPE else ""),
                       "Validation ROC-AUC": choice.set_index("Model type").at[m, "Validation ROC-AUC"],
                       "Validation balanced accuracy %": choice.set_index("Model type").at[m, "Validation balanced accuracy %"],
                       "Test balanced accuracy %": at_cut(yt, ptst, cut)["balanced_accuracy"] * 100,
                       "Test BA 95% range": f"{r['balanced_accuracy'][0] * 100:.1f} to {r['balanced_accuracy'][1] * 100:.1f}",
                       "Test ROC-AUC": fast_auc(yt, ptst)})
paper_models = pd.DataFrame(paper_rows)
show(paper_models, "Full model: the final model and the other candidates", 3)
SHEETS["Models for the paper"] = paper_models.round(5)

### 6.3 Photo timing and the safety score

The safety score describes the street as it looked when the photo was
taken. Most photos were taken after the crash, so for some crashes the
street may have changed. Here the full model with and without the
average safety score is compared again, only on test crashes whose photo
was taken within 12 months, and within 24 months, of the crash.

In [ ]:
photo_rows = []
if PHOTO:
    gap_test = data["photo_gap_months"].values[IDX["test"]]
    for limit in [12, 24, None]:
        sel = np.ones(len(yt), bool) if limit is None else (np.abs(gap_test) <= limit)
        ys = yt[sel]
        if sel.sum() < 200 or ys.sum() < 20:
            photo_rows.append({"Test crashes": f"photo within {limit} months", "Crashes": int(sel.sum()), "Note": "too few crashes"})
            continue
        p0, p1 = pred("Full", "fit+val", "test")[sel], pred("Full + safety mean", "fit+val", "test")[sel]
        b0, b1 = at_cut(ys, p0, CUTS["Full"])["balanced_accuracy"], at_cut(ys, p1, CUTS["Full + safety mean"])["balanced_accuracy"]
        rng = np.random.RandomState(RANDOM_SEED)
        d = []
        for _ in range(BOOTSTRAP_DIFFERENCES):
            i = rng.randint(0, len(ys), len(ys))
            d.append(at_cut(ys[i], p1[i], CUTS["Full + safety mean"])["balanced_accuracy"] - at_cut(ys[i], p0[i], CUTS["Full"])["balanced_accuracy"])
        photo_rows.append({"Test crashes": "all" if limit is None else f"photo within {limit} months", "Crashes": int(sel.sum()),
                           "Serious %": ys.mean() * 100, "Full BA %": b0 * 100, "Full + safety mean BA %": b1 * 100,
                           "Difference (points)": (b1 - b0) * 100,
                           "Difference 95% range": f"{np.percentile(d, 2.5) * 100:+.2f} to {np.percentile(d, 97.5) * 100:+.2f}",
                           "Full ROC-AUC": fast_auc(ys, p0), "Full + safety mean ROC-AUC": fast_auc(ys, p1)})
photo_check = pd.DataFrame(photo_rows)
show(photo_check, "Safety score by photo timing (test part):", 2) if len(photo_check) else print("Photo timing not available.")
SHEETS["Photo timing check"] = photo_check.round(5)
save_excel()

### 6.4 Charts of the final results

In [ ]:
def version_colour(v):
    return GREY if v in ("Safety score only", "Cyclist only") else (BLUE if v.startswith("Full") else LIGHT)

view = test_table.set_index("Version").reindex(list(VERSIONS))
fig, ax = plt.subplots(figsize=(DOUBLE, 0.27 * len(view) + 0.9))
ypos = np.arange(len(view))[::-1]
col = [version_colour(v) for v in view.index]
ax.hlines(ypos, view["BA low"], view["BA high"], color=col, linewidth=1.6)
ax.scatter(view["Balanced accuracy %"], ypos, color=col, s=22, zorder=3, edgecolor="white", linewidth=0.5)
for yp, v, hi in zip(ypos, view["Balanced accuracy %"], view["BA high"]):
    ax.text(hi + 0.2, yp, f"{v:.1f}", va="center", fontsize=7)
ax.axvline(50, color=GREY, linestyle=":", linewidth=0.8)
ax.set_yticks(ypos)
ax.set_yticklabels(view.index)
ax.set_xlabel("Balanced accuracy on the test part (%)\ndot = result, line = 95% range, dotted line = 50% (guessing)")
ax.set_title(f"Balanced accuracy of every version (final model: {FINAL_TYPE})")
ax.set_xlim(min(49, view["BA low"].min() - 0.5), view["BA high"].max() + 1.5)
plt.tight_layout()
save_chart(fig, "04_balanced_accuracy_versions")

roc_rows = []
fig, ax = plt.subplots(figsize=(SINGLE, SINGLE))
for name, colour, style in [("Safety score only", GREY, ":"), ("Cyclist only", GREY, "--"), ("Street", LIGHT, "-"),
                            ("Full", BLUE, "-"), ("Full + safety mean", ORANGE, "-")]:
    p_ = pred(name, "fit+val", "test")
    fpr, tpr, _ = roc_curve(yt, p_)
    ax.plot(fpr, tpr, color=colour, linestyle=style, linewidth=1.1, label=f"{name} ({fast_auc(yt, p_):.3f})")
    keep = np.linspace(0, len(fpr) - 1, min(len(fpr), 400)).astype(int)
    roc_rows += [{"Version": name, "False positive rate": fpr[j], "True positive rate": tpr[j]} for j in keep]
ax.plot([0, 1], [0, 1], color="#BBBBBB", linewidth=0.6)
ax.set_xlabel("Slight crashes wrongly flagged (share)")
ax.set_ylabel("Serious crashes caught (share)")
ax.set_title("ROC curves, test part (ROC-AUC)")
ax.legend(loc="lower right", fontsize=6.5)
ax.set_aspect("equal")
plt.tight_layout()
save_chart(fig, "05_roc_curves")
SHEETS["Chart data ROC"] = pd.DataFrame(roc_rows).round(5)

fig, ax = plt.subplots(figsize=(DOUBLE, 0.3 * len(comparisons) + 0.9))
ypos = np.arange(len(comparisons))[::-1]
ax.hlines(ypos, comparisons["Low"], comparisons["High"], color=BLUE, linewidth=1.6)
ax.scatter(comparisons["Difference (points)"], ypos, color=BLUE, s=20, zorder=3)
main_row = comparisons.index[comparisons["Main question"]]
ax.scatter(comparisons.loc[main_row, "Difference (points)"], ypos[main_row], s=70, facecolor="none", edgecolor=ORANGE, linewidth=1.2,
           zorder=4, label="Main question")
ax.axvline(0, color=GREY, linestyle="--", linewidth=0.8)
ax.set_yticks(ypos)
ax.set_yticklabels(comparisons["Comparison"])
ax.set_xlabel("Balanced accuracy with minus without the added columns (points)\ndot = result, line = 95% range")
ax.set_title("Does adding the safety score (or photo timing) change balanced accuracy?")
ax.legend(loc="lower right")
plt.tight_layout()
save_chart(fig, "06_safety_score_balanced_accuracy")

p = pred("Full", "fit+val", "test")
flag = p >= CUTS["Full"]
cm = np.array([[((~flag) & (yt == 0)).sum(), (flag & (yt == 0)).sum()], [((~flag) & (yt == 1)).sum(), (flag & (yt == 1)).sum()]])
fig, ax = plt.subplots(figsize=(SINGLE, 2.6))
ax.imshow(cm / cm.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for (r_, c_), v in np.ndenumerate(cm):
    share = v / cm[r_].sum()
    ax.text(c_, r_, f"{v:,}\n{share * 100:.1f}% of the row", ha="center", va="center", fontsize=7.5,
            color="white" if share > 0.5 else "black")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Predicted slight", "Predicted serious"])
ax.set_yticks([0, 1])
ax.set_yticklabels(["Actually slight", "Actually serious"])
ax.set_title("Full model on the test part (each row adds up to 100%)")
for sp in ax.spines.values():
    sp.set_visible(False)
plt.tight_layout()
save_chart(fig, "07_confusion_full_model")

deciles = pd.DataFrame({"p": p, "y": yt})
deciles["tenth"] = pd.qcut(deciles["p"].rank(method="first"), 10, labels=range(1, 11))
dec = deciles.groupby("tenth", observed=True)["y"].agg(["sum", "size"]).reset_index()
dec["serious %"] = dec["sum"] / dec["size"] * 100
dec[["low", "high"]] = [wilson(k, m_) for k, m_ in zip(dec["sum"], dec["size"])]
fig, ax = plt.subplots(figsize=(SINGLE, 2.6))
ax.bar(dec["tenth"].astype(int), dec["serious %"], color=BLUE, width=0.7)
ax.errorbar(dec["tenth"].astype(int), dec["serious %"], yerr=[dec["serious %"] - dec["low"], dec["high"] - dec["serious %"]],
            fmt="none", ecolor="black", elinewidth=0.6, capsize=2)
ax.axhline(yt.mean() * 100, color=ORANGE, linestyle="--", linewidth=0.8, label=f"All test crashes ({yt.mean() * 100:.1f}%)")
ax.set_xlabel("Predicted risk, tenths of the test crashes (1 = lowest)")
ax.set_ylabel("Serious or fatal (%)\nout of the crashes in the tenth")
ax.set_title("Observed severity by predicted risk")
ax.set_xticks(range(1, 11))
ax.legend(loc="upper left")
plt.tight_layout()
save_chart(fig, "08_risk_tenths")
SHEETS["Risk tenths"] = dec.rename(columns={"sum": "serious crashes", "size": "crashes", "low": "95% low", "high": "95% high"}).round(4)

## Section 7: feature impact (SHAP)

SHAP splits each crash's predicted risk into a contribution from each
feature, on the log-odds scale. A positive value pushes the risk of a
serious injury up; a negative one pushes it down. The average size of the
contributions measures how much a feature matters; their direction shows
whether higher values (or a category being present) raise or lower the
risk.

It is calculated for the full model (main training) on the test crashes:
exactly for the elastic-net logistic regression (weight x standardised value), and with the
standard tree method (TreeSHAP) for XGBoost and LightGBM, averaged over
the five seeds. If the final model is the average of the three, the SHAP
values are the average of the three models' SHAP values (an
approximation, because the models are averaged on the probability scale).
Categories appear one value per row, for example
"junctiondetail = Roundabout", so every row has a direction.

In [ ]:
def nice_name(encoded, cols):
    group, name = encoded.split("__", 1) if "__" in encoded else ("", encoded)
    if name.startswith("missingindicator_"):
        return f"{name.replace('missingindicator_', '')}: missing"
    if group.startswith("cat"):
        for c in sorted(cols, key=len, reverse=True):
            if name.startswith(c + "_"):
                return f"{c} = {name[len(c) + 1:]}"
    return name

frames, x_values = [], {}
for member in MEMBERS:
    r = RESULTS[(member, "Full", "fit+val")]
    Xd = r["X_eval"].toarray() if hasattr(r["X_eval"], "toarray") else np.asarray(r["X_eval"])
    contribs = []
    for model in r["models"]:
        if member == "XGBoost":
            c = model.get_booster().predict(xgb.DMatrix(Xd), pred_contribs=True)[:, :-1]
        elif member == "LightGBM":
            c = np.asarray(model.predict(Xd, pred_contrib=True))[:, :-1]
        else:
            mu = np.asarray(r["X_train"].mean(axis=0)).ravel()
            c = (Xd - mu) * model.coef_[0]
        contribs.append(c)
    member_names, seen = [], {}
    for e in r["names"]:
        nm = nice_name(e, r["cols"])
        seen[nm] = seen.get(nm, 0) + 1
        member_names.append(nm if seen[nm] == 1 else f"{nm} ({seen[nm]})")
    frames.append(pd.DataFrame(np.mean(contribs, axis=0), columns=member_names))
    for j, nm in enumerate(member_names):
        if nm not in x_values or member != "Elastic-net logistic regression":       # keep the unscaled values of a tree model when available
            x_values[nm] = Xd[:, j]
all_names = list(dict.fromkeys(nm for f in frames for nm in f.columns))
SHAP = (sum(f.reindex(columns=all_names, fill_value=0.0) for f in frames) / len(frames)).values
names = all_names
X_dense = np.column_stack([x_values[nm] for nm in names])
print(f"SHAP values for {len(names)} inputs, {SHAP.shape[0]:,} test crashes, model(s): {', '.join(MEMBERS)}")
rows = []
for j, nm in enumerate(names):
    s, x = SHAP[:, j], X_dense[:, j]
    binary = set(np.unique(x)).issubset({0.0, 1.0})
    if np.std(x) == 0 or np.std(s) == 0:
        direction = 0.0
    elif binary:
        direction = s[x == 1].mean() - s[x == 0].mean() if (x == 1).any() and (x == 0).any() else 0.0
    else:
        direction = spearmanr(x, s).correlation
    raw = nm.split(" = ")[0].split(": missing")[0]
    rows.append({"Input": nm, "Feature": raw, "Family": FAMILY.get(raw, "Safety score" if raw in SAFETY_COLS else ""),
                 "Mean |SHAP|": np.abs(s).mean(), "Direction": "raises risk" if direction > 0 else ("lowers risk" if direction < 0 else "none"),
                 "Kind": "yes/no" if binary else "number", "Reading": ""})
shap_table = pd.DataFrame(rows).sort_values("Mean |SHAP|", ascending=False).reset_index(drop=True)
shap_table["Share of total %"] = shap_table["Mean |SHAP|"] / shap_table["Mean |SHAP|"].sum() * 100
shap_table["Readable name"] = shap_table["Input"].map(readable_input)
def reading(r):
    if r["Direction"] == "none":
        return "no clear direction"
    up = r["Direction"] == "raises risk"
    if r["Kind"] == "yes/no":
        return f"{r['Readable name']}: {'higher' if up else 'lower'} risk of a serious injury"
    return f"Higher {r['Readable name'].lower()}: {'higher' if up else 'lower'} risk of a serious injury"
shap_table["Reading"] = shap_table.apply(reading, axis=1)
by_feature = shap_table.groupby(["Feature", "Family"], as_index=False)["Mean |SHAP|"].sum().sort_values("Mean |SHAP|", ascending=False)
by_feature["Share of total %"] = by_feature["Mean |SHAP|"] / by_feature["Mean |SHAP|"].sum() * 100
by_feature.insert(1, "Readable name", by_feature["Feature"].map(readable_feature))
shap_table = shap_table[["Input", "Readable name", "Feature", "Family", "Kind", "Mean |SHAP|", "Share of total %", "Direction", "Reading"]]
show(shap_table.head(15)[["Readable name", "Family", "Share of total %", "Direction", "Reading"]], "Top 15 inputs by impact (test part):", 2)
show(by_feature.head(15).reset_index(drop=True), "\nTop 15 features (all categories of a feature added together):", 3)
SHEETS["SHAP inputs"] = shap_table.round(6)
SHEETS["SHAP features"] = by_feature.round(6)
save_excel()

TOP = min(15, len(shap_table))
top = shap_table.head(TOP).iloc[::-1]
fig, ax = plt.subplots(figsize=(DOUBLE, 4.2))
signed = np.where(top["Direction"] == "raises risk", 1, np.where(top["Direction"] == "lowers risk", -1, 0)) * top["Mean |SHAP|"]
ax.barh(top["Readable name"], signed, color=[RAISES if v > 0 else LOWERS for v in signed], height=0.65)
ax.axvline(0, color=GREY, linewidth=0.8)
ax.set_xlabel("Average impact on the log-odds of a serious or fatal injury\n(bar length = size of the impact; right = raises the risk, left = lowers it)")
ax.set_title(f"The {TOP} most important inputs of the full model, with direction")
handles = [plt.Rectangle((0, 0), 1, 1, color=RAISES), plt.Rectangle((0, 0), 1, 1, color=LOWERS)]
ax.legend(handles, ["Raises the risk", "Lowers the risk"], loc="lower right")
plt.tight_layout()
save_chart(fig, "09_shap_top15_signed")

order = [shap_table.index[i] for i in range(TOP)]
idx_cols = [names.index(shap_table.at[i, "Input"]) for i in order]
fig, ax = plt.subplots(figsize=(DOUBLE, 4.6))
rng = np.random.RandomState(RANDOM_SEED)
for row, j in enumerate(idx_cols[::-1]):
    sv, x = SHAP[:, j], X_dense[:, j]
    colour_value = rankdata(x) / len(x)
    sel = rng.choice(len(sv), size=min(len(sv), 3000), replace=False)
    ax.scatter(sv[sel], row + rng.uniform(-0.3, 0.3, len(sel)), c=colour_value[sel], cmap="coolwarm", s=2.5, alpha=0.6, linewidths=0)
ax.set_yticks(range(TOP))
ax.set_yticklabels([shap_table.at[i, "Readable name"] for i in order][::-1])
ax.axvline(0, color=GREY, linewidth=0.8)
ax.set_xlabel("SHAP value: impact on the log-odds of a serious or fatal injury\n(one dot per test crash)")
ax.set_title("How each input moves the predicted risk (colour = value of the input)")
sm = plt.cm.ScalarMappable(cmap="coolwarm")
cb = plt.colorbar(sm, ax=ax, fraction=0.025, pad=0.01)
cb.set_ticks([0, 1])
cb.set_ticklabels(["low value", "high value"])
plt.tight_layout()
save_chart(fig, "10_shap_summary_dots")

numeric_top = [i for i in shap_table.index if shap_table.at[i, "Kind"] == "number"][:4]
test_rows = data.iloc[IDX["test"]]
if numeric_top:
    fig, axes = plt.subplots(1, len(numeric_top), figsize=(DOUBLE, 2.3), squeeze=False)
    for ax, i in zip(axes[0], numeric_top):
        nm = shap_table.at[i, "Input"]
        j = names.index(nm)
        raw = test_rows[nm].values if nm in test_rows.columns else X_dense[:, j]
        ax.scatter(raw, SHAP[:, j], s=2.5, alpha=0.35, color=BLUE, linewidths=0)
        ax.axhline(0, color=GREY, linewidth=0.8)
        ax.set_xlabel("\n".join(re.findall(".{1,28}(?:\\s|$)", shap_table.at[i, "Readable name"])), fontsize=7)
        ax.set_ylabel("SHAP value" if ax is axes[0][0] else "")
    fig.suptitle("How the impact changes along the most important number inputs", x=0.01, ha="left", fontsize=9, fontweight="bold")
    plt.tight_layout()
    save_chart(fig, "11_shap_dependence")

# SHAP values of the 50 most important inputs for every test crash, with the inputs' values, for later charts
top50 = shap_table.head(50)["Input"].tolist()
shap_export = pd.DataFrame({"collision_id": test_rows["collision_id"].values})
for nm in top50:
    j = names.index(nm)
    shap_export[f"SHAP | {readable_input(nm)}"] = SHAP[:, j]
    shap_export[f"value | {readable_input(nm)}"] = X_dense[:, j]
shap_export.to_csv(os.path.join(OUTPUT_FOLDER, "shap_values_test.csv"), index=False)
print(f"Saved shap_values_test.csv: {len(shap_export):,} test crashes x the 50 most important inputs (SHAP value and input value).")

## Section 8: severity breakdowns and maps

A description of severity for the paper, using all crashes from the start
date. For each group:

- **Share of all crashes**: the group's crashes out of all crashes. These
  shares add up to 100%.
- **Serious or fatal within the group**: out of the crashes in that group,
  the share that were serious or fatal, with its 95% range. These shares do
  not add up to 100%: each one is out of its own group.

### 8.1 Severity by group

In [ ]:
allc = data.iloc[:n].copy()
part_label = np.empty(n, dtype=object)
part_label[IDX["fit"]], part_label[IDX["val"]], part_label[IDX["test"]] = "fitting", "validation", "test"
allc["part"] = part_label
INNER = ["camden", "cityoflondon", "hackney", "hammersmith", "haringey", "islington", "kensington", "lambeth", "lewisham",
         "newham", "southwark", "towerhamlets", "wandsworth", "westminster"]
norm = allc["borough"].fillna("").astype(str).str.lower().str.replace("&", "and").str.replace(r"[^a-z]", "", regex=True)
allc["Inner or Outer London"] = np.where(norm.apply(lambda b_: any(k in b_ for k in INNER)), "Inner London", "Outer London")
speed = pd.to_numeric(allc["speedlimit"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
allc["Speed limit"] = pd.cut(speed, [0, 20, 30, 200], labels=["20 mph or less", "30 mph", "40 mph or more"]).astype(object)
allc["Season"] = allc["accident_dt"].dt.month.map(season_of)
if "date_is_weekend" in allc.columns:
    allc["Weekday or weekend"] = np.where(pd.to_numeric(allc["date_is_weekend"], errors="coerce") == 1, "Weekend", "Weekday")
if "IMDDecile" in allc.columns:
    allc["Deprivation decile"] = pd.to_numeric(allc["IMDDecile"], errors="coerce").map(lambda v: f"{int(v):02d}" if pd.notna(v) else np.nan)
allc["Perceived safety fifth"] = pd.qcut(allc["safety_score_mean"].rank(method="first"), 5,
                                         labels=["1 (least safe-looking)", "2", "3", "4", "5 (most safe-looking)"]).astype(object)
for col, label in [("borough", "Borough"), ("roadtype", "Road type"), ("junctiondetail", "Junction type"),
                   ("lightconditionsbanded", "Light conditions"), ("date_hour_bin", "Time of day"), ("age_group", "Cyclist age group"),
                   ("casualty_sex", "Cyclist sex")]:
    if col in allc.columns:
        allc[label] = allc[col].map(lambda v: readable_category(v) if pd.notna(v) else np.nan)
BREAKDOWNS = ["Borough", "Inner or Outer London", "Speed limit", "Road type", "Junction type", "Light conditions", "Time of day",
              "Weekday or weekend", "Season", "Cyclist age group", "Cyclist sex", "Deprivation decile", "Perceived safety fifth"]
ORDERED = {"Cyclist age group": ["under 16", "16-24", "25-34", "35-44", "45-54", "55-64", "65+", "missing"],
           "Season": list(SEASON_COLOURS), "Speed limit": ["20 mph or less", "30 mph", "40 mph or more", "missing"]}
overall = allc["target"].mean() * 100

def severity_table(col):
    g = allc[col].astype(object).where(allc[col].notna(), "missing").astype(str)
    tab = allc.groupby(g)["target"].agg(["size", "sum"]).rename(columns={"size": "Crashes", "sum": "Serious or fatal"}).reset_index()
    tab = tab.rename(columns={tab.columns[0]: "Group"})
    tab["Share of all crashes %"] = tab["Crashes"] / tab["Crashes"].sum() * 100
    tab["Serious or fatal within the group %"] = tab["Serious or fatal"] / tab["Crashes"] * 100
    tab[["95% low", "95% high"]] = [wilson(k, m_) for k, m_ in zip(tab["Serious or fatal"], tab["Crashes"])]
    if col in ORDERED:
        tab["_o"] = tab["Group"].map(lambda x: ORDERED[col].index(x) if x in ORDERED[col] else 99)
        tab = tab.sort_values("_o").drop(columns="_o")
    elif col in ("Deprivation decile", "Perceived safety fifth"):
        tab = tab.sort_values("Group")
    else:
        tab = tab.sort_values("Crashes", ascending=False)
    return tab.reset_index(drop=True)

def severity_chart(tab, title, fname, min_crashes=30):
    tab = tab[tab["Crashes"] >= min_crashes].iloc[::-1]
    h = 0.24 * len(tab) + 1.1
    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, h), sharey=True, gridspec_kw={"width_ratios": [1, 1.25], "wspace": 0.08})
    yy = np.arange(len(tab))
    axes[0].barh(yy, tab["Share of all crashes %"], color=LIGHT, height=0.65)
    for y_, v, c in zip(yy, tab["Share of all crashes %"], tab["Crashes"]):
        axes[0].text(v + axes[0].get_xlim()[1] * 0.0, y_, f" {v:.1f}% (n = {c:,})", va="center", fontsize=6.5)
    axes[0].set_xlim(0, tab["Share of all crashes %"].max() * 1.75)
    axes[0].set_yticks(yy)
    axes[0].set_yticklabels(tab["Group"])
    axes[0].set_xlabel("Share of all crashes (%)\nthe groups add up to 100%")
    axes[1].hlines(yy, tab["95% low"], tab["95% high"], color=BLUE, linewidth=1.4)
    axes[1].scatter(tab["Serious or fatal within the group %"], yy, color=BLUE, s=16, zorder=3)
    axes[1].axvline(overall, color=ORANGE, linestyle="--", linewidth=0.8)
    axes[1].set_xlabel("Serious or fatal (%) out of the group's crashes\ndot = share, line = 95% range")
    axes[1].set_title(f"dashed line = all crashes ({overall:.1f}%)", fontsize=7, fontweight="normal", color=ORANGE, loc="right")
    fig.suptitle(title, x=0.01, ha="left", fontsize=9, fontweight="bold")
    plt.tight_layout()
    save_chart(fig, fname)

period_text = f"{allc['accident_dt'].min():%b %Y} to {allc['accident_dt'].max():%b %Y}"
all_tabs = []
for k, col in enumerate(BREAKDOWNS):
    if col not in allc.columns or allc[col].isna().all():
        print(f"{col}: not available, skipped")
        continue
    tab = severity_table(col)
    print(f"\n{col}")
    display(tab.round(1))
    severity_chart(tab, f"{col}: share of crashes and serious or fatal share ({period_text})", f"12_{k + 1:02d}_severity_{re.sub(r'[^a-z]+', '_', col.lower()).strip('_')}")
    all_tabs.append(tab.assign(Breakdown=col))
breakdowns = pd.concat(all_tabs, ignore_index=True)
SHEETS["Breakdowns"] = breakdowns[["Breakdown", "Group", "Crashes", "Share of all crashes %", "Serious or fatal",
                                   "Serious or fatal within the group %", "95% low", "95% high"]].round(3)
save_excel()

### 8.2 Severity across the most important features

The six most important features of the final model (by SHAP), and the
serious or fatal share across their values: fifths of the crashes for
number features, the most common values for categories.

In [ ]:
top_features = [f for f in by_feature["Feature"] if f in allc.columns][:6]
feature_tabs = []
fig, axes = plt.subplots(2, 3, figsize=(DOUBLE, 5.2))
for ax, f in zip(axes.ravel(), top_features):
    v = allc[f]
    if pd.api.types.is_numeric_dtype(v) and v.nunique() > 6:
        try:
            groups = pd.qcut(v.rank(method="first"), 5, labels=False)
            edges_ = [v[groups == q].agg(["min", "max"]) for q in range(5)]
            labels_ = [f"{e['min']:.3g} to {e['max']:.3g}" for e in edges_]
            g = groups.map(lambda q: labels_[int(q)] if pd.notna(q) else "missing")
            order_ = labels_ + (["missing"] if g.eq("missing").any() else [])
        except ValueError:
            g, order_ = v.astype(str), None
    else:
        g = v.astype(object).where(v.notna(), "missing").map(lambda x: readable_category(x) if x != "missing" else "missing")
        common = g.value_counts().index[:7]
        g = g.where(g.isin(common), "other")
        order_ = list(g.value_counts().index)
    tab = allc.groupby(g)["target"].agg(["size", "sum"])
    tab = tab.reindex([o for o in (order_ or tab.index) if o in tab.index])
    pct = tab["sum"] / tab["size"] * 100
    ci = np.array([wilson(k, m_) for k, m_ in zip(tab["sum"], tab["size"])])
    yy = np.arange(len(tab))[::-1]
    ax.hlines(yy, ci[:, 0], ci[:, 1], color=BLUE, linewidth=1.2)
    ax.scatter(pct, yy, color=BLUE, s=12, zorder=3)
    ax.axvline(overall, color=ORANGE, linestyle="--", linewidth=0.7)
    ax.set_yticks(yy)
    ax.set_yticklabels([str(x)[:26] for x in tab.index], fontsize=6.5)
    ax.set_title("\n".join(re.findall(".{1,34}(?:\\s|$)", readable_feature(f))).strip(), fontsize=7.5)
    ax.set_xlabel("Serious or fatal (%)", fontsize=7)
    feature_tabs.append(pd.DataFrame({"Feature": f, "Readable name": readable_feature(f), "Group": tab.index, "Crashes": tab["size"].values,
                                      "Share of all crashes %": tab["size"].values / len(allc) * 100, "Serious or fatal within the group %": pct.values,
                                      "95% low": ci[:, 0], "95% high": ci[:, 1]}))
for ax in axes.ravel()[len(top_features):]:
    ax.set_visible(False)
fig.suptitle(f"Serious or fatal share across the six most important features (dashed line = all crashes, {overall:.1f}%)",
             x=0.01, ha="left", fontsize=9, fontweight="bold")
plt.tight_layout()
save_chart(fig, "13_severity_top_features")
SHEETS["Top feature severity"] = pd.concat(feature_tabs, ignore_index=True).round(3) if feature_tabs else pd.DataFrame()
save_excel()

### 8.3 Maps

Crash locations are in the British National Grid. The hexagon maps need
no extra file. The borough maps use the boundaries from the London
Datastore (`London_Boroughs.gpkg`, Ordnance Survey data); if the file or
the geopandas package is missing, they are skipped.

In [ ]:
CREDIT = "Contains National Statistics and Ordnance Survey data. Crown copyright and database right."
boroughs = None
try:
    import geopandas as gpd
    if os.path.exists(BOROUGHS_GPKG):
        boroughs = gpd.read_file(BOROUGHS_GPKG)
        boroughs = boroughs.set_crs(27700) if boroughs.crs is None else (boroughs if boroughs.crs.to_epsg() == 27700 else boroughs.to_crs(27700))
        preferred = [c for c in boroughs.columns if c.lower() in ("name", "lad22nm", "lad21nm", "lad23nm", "borough", "lad_name", "nm")]
        text_cols = [c for c in boroughs.columns if c != "geometry" and boroughs[c].dtype == object]
        NAME_COL = (preferred or text_cols)[0]
        print(f"Borough boundaries: {len(boroughs)} areas (name column '{NAME_COL}').")
except Exception as error:
    print(f"Borough boundaries not available: {str(error)[:150]}")

xy_ok = allc["easting"].notna() & allc["northing"].notna()
ex, ny, yy_ = allc.loc[xy_ok, "easting"].values, allc.loc[xy_ok, "northing"].values, allc.loc[xy_ok, "target"].values

def base_map(ax):
    if boroughs is not None:
        boroughs.boundary.plot(ax=ax, color="#555555", linewidth=0.3)
    ax.set_aspect("equal")
    ax.set_axis_off()

fig, axes = plt.subplots(1, 2, figsize=(DOUBLE, 3.4))
hb = axes[0].hexbin(ex, ny, gridsize=45, mincnt=1, cmap=MAP_CMAP, bins="log", linewidths=0)
base_map(axes[0])
cb = fig.colorbar(hb, ax=axes[0], fraction=0.035, pad=0.01)
cb.set_label("Crashes per hexagon (log)")
axes[0].set_title("Where the crashes happened")
hb2 = axes[1].hexbin(ex, ny, C=yy_ * 100, reduce_C_function=np.mean, gridsize=30, mincnt=40, cmap=MAP_CMAP, linewidths=0)
base_map(axes[1])
cb2 = fig.colorbar(hb2, ax=axes[1], fraction=0.035, pad=0.01)
cb2.set_label("Serious or fatal (%)")
axes[1].set_title("Serious or fatal share (hexagons with 40+ crashes)")
fig.text(0.01, 0.01, CREDIT, fontsize=5.5, color=GREY)
plt.tight_layout()
save_chart(fig, "14_maps_hexagons")

test_df = allc.iloc[IDX["test"]].copy()
test_df["predicted risk percentile"] = rankdata(pred("Full", "fit+val", "test")) / len(IDX["test"]) * 100
if boroughs is not None:
    try:
        pts = gpd.GeoDataFrame(allc.loc[xy_ok, ["target", "safety_score_mean"]].copy(),
                               geometry=gpd.points_from_xy(ex, ny), crs=27700)
        joined = gpd.sjoin(pts, boroughs[[NAME_COL, "geometry"]], how="left", predicate="within")
        stats = joined.groupby(NAME_COL).agg(crashes=("target", "size"), serious=("target", "sum"), safety=("safety_score_mean", "mean")).reset_index()
        stats["Serious or fatal %"] = stats["serious"] / stats["crashes"] * 100
        stats["Share of all crashes %"] = stats["crashes"] / stats["crashes"].sum() * 100
        tpts = gpd.GeoDataFrame(test_df[["predicted risk percentile"]].copy(),
                                geometry=gpd.points_from_xy(test_df["easting"], test_df["northing"]), crs=27700)
        tjoin = gpd.sjoin(tpts, boroughs[[NAME_COL, "geometry"]], how="left", predicate="within")
        stats = stats.merge(tjoin.groupby(NAME_COL)["predicted risk percentile"].mean().rename("Average predicted risk percentile (test)").reset_index(),
                            on=NAME_COL, how="left")
        mapdf = boroughs.merge(stats, on=NAME_COL, how="left")
        panels = [("Share of all crashes %", "Share of all crashes (%)"), ("Serious or fatal %", "Serious or fatal (%) out of the borough's crashes"),
                  ("safety", "Average perceived safety (Paper 1 score)"), ("Average predicted risk percentile (test)", "Average predicted risk (percentile, test crashes)")]
        fig, axes = plt.subplots(2, 2, figsize=(DOUBLE, 6.0))
        for ax, (col, label) in zip(axes.ravel(), panels):
            mapdf.plot(column=col, cmap=MAP_CMAP, legend=True, ax=ax, edgecolor="white", linewidth=0.3,
                       legend_kwds={"shrink": 0.6, "label": label}, missing_kwds={"color": "#EEEEEE"})
            ax.set_axis_off()
            ax.set_title(label.split(" (")[0])
        fig.text(0.01, 0.01, CREDIT, fontsize=5.5, color=GREY)
        plt.tight_layout()
        save_chart(fig, "15_maps_boroughs")
        SHEETS["Borough map data"] = stats.rename(columns={"crashes": "Crashes", "serious": "Serious or fatal",
                                                           "safety": "Average perceived safety"}).round(4)
    except Exception as error:
        print(f"Borough maps skipped: {str(error)[:200]}")

fig, ax = plt.subplots(figsize=(DOUBLE * 0.75, 3.8))
order_pts = test_df.sort_values("predicted risk percentile")
sc = ax.scatter(order_pts["easting"], order_pts["northing"], c=order_pts["predicted risk percentile"], cmap="coolwarm", s=1.5, alpha=0.8, linewidths=0)
base_map(ax)
cb = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.01)
cb.set_label("Predicted risk of a serious injury (percentile)")
ax.set_title("Test crashes coloured by the full model's predicted risk")
fig.text(0.01, 0.01, CREDIT, fontsize=5.5, color=GREY)
plt.tight_layout()
save_chart(fig, "16_map_predicted_risk")
hex_rows = pd.DataFrame({"easting": ex, "northing": ny, "serious_or_fatal": yy_})
save_excel()

### 8.4 Files for further charts

Two files hold everything needed to draw new breakdowns or maps without
running the notebook again:

- `crashes_for_breakdowns.csv`: every crash from the start date with its
  breakdown groups, location, safety score and outcome.
- `test_crashes_with_predictions.csv`: every test crash with the same
  columns, plus each version's predicted risk and whether it was flagged.

In [ ]:
group_cols = [c for c in BREAKDOWNS if c in allc.columns]
base_cols = ["collision_id", "accident_dt", "part", "easting", "northing", "safety_score_mean", "age"] + PHOTO
crash_file = allc[base_cols + group_cols].copy()
crash_file["casualty_severity"] = allc[OUTCOME_COLUMN].values
crash_file["serious_or_fatal"] = allc["target"].values
crash_file.to_csv(os.path.join(OUTPUT_FOLDER, "crashes_for_breakdowns.csv"), index=False)
test_file = crash_file.iloc[IDX["test"]].copy().reset_index(drop=True)
for name in VERSIONS:
    p_ = pred(name, "fit+val", "test")
    test_file[f"risk | {name}"] = p_
    test_file[f"flagged | {name}"] = (p_ >= CUTS[name]).astype(int)
test_file.to_csv(os.path.join(OUTPUT_FOLDER, "test_crashes_with_predictions.csv"), index=False)
print(f"Saved crashes_for_breakdowns.csv ({len(crash_file):,} crashes) and test_crashes_with_predictions.csv ({len(test_file):,} crashes).")

## Section 9: robustness over two time periods

The whole method (selecting the street features again with the same
option, training the final model, choosing the cut-off on the period just
before) is repeated on two test periods of the same length, each time
training only on earlier crashes:

- **Period 1**: the validation part.
- **Period 2**: the test part, with the model trained on the fitting part only.

This shows whether the results hold at different times, not only in the
main test.

In [ ]:
t0 = time.time()
LV = len(IDX["val"])
period_rows = []
for k, label, start in [(1, "Period 1 (the validation part)", fit_end), (2, "Period 2 (the test part)", val_end)]:
    IDX[f"p{k}_fit"], IDX[f"p{k}_cut"] = np.arange(0, start - LV), np.arange(start - LV, start)
    IDX[f"p{k}_test"] = np.arange(start, min(start + LV, n))
    IDX[f"p{k}_all"] = np.concatenate([IDX[f"p{k}_cut"], IDX[f"p{k}_test"]])
    train_versions(f"p{k}_fit", f"p{k}_all", f"p{k}_fit", IDX[f"p{k}_fit"], fixed=FIXED_K,
                   only=["Street", "Full", "Full + safety mean"], suffix=f" [p{k}]", keep=())
    y_cut, y_k = y_all[IDX[f"p{k}_cut"]], y_all[IDX[f"p{k}_test"]]
    res = {}
    for nm in ["Street", "Full", "Full + safety mean"]:
        pc, pt = pred(f"{nm} [p{k}]", f"p{k}_fit", f"p{k}_cut"), pred(f"{nm} [p{k}]", f"p{k}_fit", f"p{k}_test")
        cut = best_cut(y_cut, pc)
        r = ranges(y_k, pt, cut, 300)
        res[nm] = (at_cut(y_k, pt, cut)["balanced_accuracy"] * 100, r["balanced_accuracy"] * 100, fast_auc(y_k, pt))
    part = data.iloc[IDX[f"p{k}_test"]]
    period_rows.append({"Test period": label, "From": part["accident_dt"].min().strftime("%Y-%m"), "To": part["accident_dt"].max().strftime("%Y-%m"),
                        "Training crashes": len(IDX[f"p{k}_fit"]),
                        "Street BA %": res["Street"][0], "Street BA range": f"{res['Street'][1][0]:.1f} to {res['Street'][1][1]:.1f}",
                        "Full BA %": res["Full"][0], "Full BA range": f"{res['Full'][1][0]:.1f} to {res['Full'][1][1]:.1f}",
                        "Full + safety mean BA %": res["Full + safety mean"][0],
                        "Street ROC-AUC": res["Street"][2], "Full ROC-AUC": res["Full"][2], "Full + safety mean ROC-AUC": res["Full + safety mean"][2]})
periods = pd.DataFrame(period_rows)
show(periods, digits=3)
SHEETS["Time periods"] = periods.round(5)
save_excel()

fig, ax = plt.subplots(figsize=(DOUBLE * 0.6, 2.8))
x = np.arange(len(periods))
for colname, colour, label in [("Street BA %", LIGHT, "Street"), ("Full BA %", BLUE, "Full"), ("Full + safety mean BA %", ORANGE, "Full + safety mean")]:
    ax.plot(x, periods[colname], marker="o", color=colour, label=label)
ax.set_xticks(x)
ax.set_xticklabels([f"{r['From']} to {r['To']}" for _, r in periods.iterrows()])
ax.set_ylabel("Balanced accuracy (%)")
ax.set_title("Balanced accuracy in two test periods")
ax.legend(frameon=False)
save_chart(fig, "17_time_periods")
print(f"Time: {(time.time() - t0) / 60:.1f} min")

## Section 10: decision log, Excel file and summary

In [ ]:
# ============ 10.1 Feature journey: the steps first, then every column ============
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment

journey1 = pd.read_excel(JOURNEY_XLSX, sheet_name="Every column").rename(
    columns={"column": "Feature", "step": "Stage 1 step", "decision": "Stage 1 decision", "reason": "Stage 1 reason"})
journey1["Stage 1 step"] = journey1["Stage 1 step"].astype(str).str.strip()
journey1["Stage 1 reason"] = journey1["Stage 1 reason"].fillna("").astype(str)
built_in_stage1 = journey1["Stage 1 reason"].str.contains("Built in this notebook")
raw_count = int((~built_in_stage1).sum())
journey1.loc[built_in_stage1, "Stage 1 reason"] = "Built in stage 1 from other columns"
journey1.loc[(journey1["Stage 1 decision"] == "Kept as a feature") & (journey1["Stage 1 reason"] == ""), "Stage 1 reason"] = \
    "Describes the place or the conditions before the crash"

STEP_INFO = {
    "4.1": ("Crash description", "Describes the crash itself or the people and vehicles in it, so it is only known after the crash"),
    "4.2": ("Identifiers, dates and free text", "Labels and raw dates, not properties of the place"),
    "4.3": ("Weather possibly after the crash", "Weather readings that can be from after the crash; only readings at least 1 hour before are kept"),
    "4.4": ("Data processing details", "Describes how values were matched or processed, not the street"),
    "4.5": ("Year columns", "A year is never seen again in later periods and follows the change in recording"),
    "4.6": ("Exact location", "Coordinates that pin down one place, not the kind of place"),
    "6": ("Duplicates and time-tracking", "Copies of a kept column, or columns that follow the crash date"),
}
FAMILY_GROUP = {"Points of interest": "Place and surroundings", "Cyclist exposure": "Place and surroundings",
                "Noise": "Place and surroundings", "Deprivation and high street": "Place and surroundings",
                "Crash history counts": "Crash history of the area", "Earlier severity ratios": "Crash history of the area",
                "Weather readings": "Weather before the crash", "Road and crash record": "Road and junction",
                "Road feature pairs": "Road and junction", "Calendar": "Time of the crash", "Cyclist": "Cyclist",
                "Safety score": "Perceived safety score (Paper 1)"}
ELIGIBLE = BASE_ALL + CYCLIST + SAFETY_COLS
MEMBER_STREET = {m: street_for(m, "fit", IDX["fit"]) for m in MEMBERS}
corrected = dict(zip(fix_table["feature"], fix_table["values set to missing"])) if len(fix_table) else {}
assoc_tab = STATS["fit"]["table"]
shap_share = by_feature.set_index("Feature")["Share of total %"] if "by_feature" in globals() else pd.Series(dtype=float)

def family_of(feature, step):
    if feature in SAFETY_COLS:
        return "Safety score"
    if feature in CYCLIST:
        return "Cyclist"
    if feature in FAMILY and isinstance(FAMILY[feature], str):
        return FAMILY[feature]
    if feature == OUTCOME_COLUMN:
        return "Outcome"
    return STEP_INFO[step][0] if step in STEP_INFO else "Not a feature"

def stage2_of(feature):
    if feature in EXCLUDED_COLUMNS:
        return "Excluded in stage 2", EXCLUDED_COLUMNS[feature]
    if feature in CYCLIST:
        return "Used in the full model", "The cyclist's age, sex and age group are known before the crash"
    if feature in SAFETY_COLS:
        return "Tested in the safety versions", "Perceived safety score from Paper 1, added to test whether it improves the prediction"
    if feature in BASE_ALL:
        used = [m for m in MEMBERS if feature in MEMBER_STREET[m]]
        if used:
            return "Used in the final street and full models", f"Selected by: {', '.join(used)}"
        return "Eligible, not selected", "Its link with severity in the fitting part was below the selection threshold"
    if feature == OUTCOME_COLUMN:
        return "Outcome", "What the model predicts (serious or fatal vs slight)"
    if feature in ("collision_id", "accident_dt"):
        return "Kept for identification", "Used to join the data and to split it by date, never as an input"
    return "Not used (removed in stage 1)", ""

def correction_of(feature):
    if feature in ("junctioncontrol", "cx_junctioncontrol_speed", "prior_ratio_junctioncontrol"):
        return "Empty junction control at crashes not at a junction set to 'Not at junction' (and the pair and ratio rebuilt)"
    if corrected.get(feature, 0) > 0:
        return f"{corrected[feature]:,} values set to missing: one of its source fields was missing"
    return ""

def missing_rule_of(feature):
    if feature not in ELIGIBLE:
        return ""
    if feature in SAFETY_COLS:
        return "No missing values (crashes without a score left out)"
    return "Filled with the median or 'missing', plus a yes/no missing marker" if marker_allowed(feature) else \
           "Filled with the median or the most common value, no marker"

rows = []
for _, r in journey1.iterrows():
    rows.append({"Feature": r["Feature"], "Origin": "Built in stage 1" if built_in_stage1.loc[_] else "Raw file",
                 "Stage 1 step": r["Stage 1 step"], "Stage 1 decision": r["Stage 1 decision"], "Stage 1 reason": r["Stage 1 reason"]})
for f in CYCLIST + SAFETY_COLS:
    if f not in set(journey1["Feature"]):
        rows.append({"Feature": f, "Origin": "Built in stage 2" if f == "age_group" else "Built after stage 1 (photos)",
                     "Stage 1 step": "", "Stage 1 decision": "", "Stage 1 reason": ""})
journey = pd.DataFrame(rows)
journey["Family"] = [family_of(f, s) for f, s in zip(journey["Feature"], journey["Stage 1 step"])]
journey["Family group"] = journey["Family"].map(FAMILY_GROUP).fillna("Not an input")
journey["Readable name"] = journey["Feature"].map(readable_feature)
s2 = journey["Feature"].map(stage2_of)
journey["Stage 2 decision"] = s2.map(lambda t: t[0])
journey["Stage 2 reason"] = s2.map(lambda t: t[1])
journey["Eligible pre-crash input"] = np.where(journey["Feature"].isin(ELIGIBLE), "Yes", "No")
journey["Changed in stage 2"] = journey["Feature"].map(correction_of)
journey["Missing values"] = journey["Feature"].map(missing_rule_of)
journey["In final street model"] = np.where(journey["Feature"].isin(STREET_MAIN), "Yes", "No")
journey["In final full model"] = np.where(journey["Feature"].isin(STREET_MAIN + CYCLIST), "Yes", "No")
for m in MEMBERS:
    journey[f"Used by {m}"] = np.where(journey["Feature"].isin(MEMBER_STREET[m] + CYCLIST), "Yes", "No")
journey["|Correlation| with severity (fitting part)"] = journey["Feature"].map(assoc_tab["|correlation|"]).round(4)
journey["Mutual information minus chance (fitting part)"] = journey["Feature"].map(assoc_tab["relevance (MI minus chance)"]).round(5)
journey["Share of SHAP importance % (full model)"] = journey["Feature"].map(shap_share).round(2)
journey["In the data file sent"] = np.where(journey["Feature"].isin(ELIGIBLE + [OUTCOME_COLUMN]), "Yes", "No")
order_cols = ["Feature", "Readable name", "Family group", "Family", "Origin", "Eligible pre-crash input", "Stage 1 step", "Stage 1 decision",
              "Stage 1 reason", "Stage 2 decision", "Stage 2 reason", "Changed in stage 2", "Missing values", "In final street model",
              "In final full model"] + [f"Used by {m}" for m in MEMBERS] + [
              "|Correlation| with severity (fitting part)", "Mutual information minus chance (fitting part)",
              "Share of SHAP importance % (full model)", "In the data file sent"]
group_order = list(dict.fromkeys(list(FAMILY_GROUP.values()) + ["Not an input"]))
journey = journey[order_cols].assign(_g=journey["Family group"].map(group_order.index)).sort_values(
    ["_g", "Family", "Feature"]).drop(columns="_g").reset_index(drop=True)

# ---- the steps, with counts ----
steps, running = [], raw_count
steps.append({"Step": 1, "Stage": "Raw data", "What happens": "Columns in the raw crash file", "Removed": "", "Added": "",
              "Columns after the step": running, "Reason": "One row per collision (the most seriously injured cyclist)", "Examples": ""})
set_aside = journey1[journey1["Stage 1 decision"].astype(str).str.startswith("Kept, not a feature")]
running -= len(set_aside)
steps.append({"Step": 2, "Stage": "Stage 1", "What happens": "Set aside: outcome, identifiers, date, age, sex and check columns",
              "Removed": len(set_aside), "Added": "", "Columns after the step": running,
              "Reason": "Needed later (outcome, splitting by date, the full model's cyclist features), never as street inputs",
              "Examples": ", ".join(set_aside["Feature"].head(5))})
k = 3
for step, (title, reason) in STEP_INFO.items():
    if step == "6":
        built = journey1[built_in_stage1]
        running += len(built)
        steps.append({"Step": k, "Stage": "Stage 1", "What happens": "Built: calendar features, road feature pairs, earlier-severity ratios, wind sine and cosine",
                      "Removed": "", "Added": len(built), "Columns after the step": running,
                      "Reason": "Describe the place and the conditions in a form the models can use", "Examples": ", ".join(built["Feature"].head(5))})
        k += 1
    removed = journey1[(journey1["Stage 1 step"] == step) & (journey1["Stage 1 decision"] == "Removed")]
    running -= len(removed)
    steps.append({"Step": k, "Stage": "Stage 1", "What happens": f"Removed: {title.lower()}", "Removed": len(removed), "Added": "",
                  "Columns after the step": running, "Reason": reason, "Examples": ", ".join(removed["Feature"].head(5))})
    k += 1
steps.append({"Step": k, "Stage": "Stage 1", "What happens": "Candidate street features after stage 1", "Removed": "", "Added": "",
              "Columns after the step": len(CANDIDATES), "Reason": "Every remaining column describes the place or the conditions before the crash", "Examples": ""})
k += 1
steps.append({"Step": k, "Stage": "Stage 2", "What happens": "Excluded: ID, processing detail and road numbers", "Removed": len([c for c in EXCLUDED_COLUMNS if c in CANDIDATES]), "Added": "",
              "Columns after the step": len(BASE_ALL), "Reason": "Not properties of the kind of street (a label, a processing detail, names of one specific road)",
              "Examples": ", ".join(c for c in EXCLUDED_COLUMNS if c in CANDIDATES)})
k += 1
n_changed = sum(1 for f in BASE_ALL if correction_of(f))
steps.append({"Step": k, "Stage": "Stage 2", "What happens": "Corrected (not removed): missing values hidden in pairs and ratios; junction control",
              "Removed": "", "Added": "", "Columns after the step": len(BASE_ALL),
              "Reason": "A pair or ratio counts as missing when a source field is missing; 'Not at junction' replaces empty junction control at non-junction crashes",
              "Examples": f"{n_changed} features changed"})
k += 1
steps.append({"Step": k, "Stage": "Stage 2", "What happens": "Added: the cyclist's age, sex and age group (full model only)", "Removed": "", "Added": len(CYCLIST),
              "Columns after the step": len(BASE_ALL) + len(CYCLIST), "Reason": "Known before the crash; age is a known predictor of injury severity",
              "Examples": ", ".join(CYCLIST)})
k += 1
steps.append({"Step": k, "Stage": "Stage 2", "What happens": "Added for testing: perceived safety score columns (safety versions only)", "Removed": "", "Added": len(SAFETY_COLS),
              "Columns after the step": len(ELIGIBLE), "Reason": "The paper's question: does the safety score improve the prediction?",
              "Examples": ", ".join(SAFETY_COLS[:4])})
k += 1
steps.append({"Step": k, "Stage": "Result", "What happens": "Eligible pre-crash inputs (the columns in the data file sent)", "Removed": "", "Added": "",
              "Columns after the step": len(ELIGIBLE), "Reason": f"{len(BASE_ALL)} street features + {len(CYCLIST)} cyclist features + {len(SAFETY_COLS)} safety score columns",
              "Examples": ""})
k += 1
for m in MEMBERS:
    steps.append({"Step": k, "Stage": "Stage 2", "What happens": f"Selected for {m} ('{BEST[m]['selection']}')", "Removed": "", "Added": "",
                  "Columns after the step": len(MEMBER_STREET[m]), "Reason": "Chosen by the search among 10 selection options, using the fitting part only",
                  "Examples": ", ".join(MEMBER_STREET[m][:4])})
    k += 1
steps.append({"Step": k, "Stage": "Result", "What happens": "Final street model (all street features used by any of the final models)", "Removed": "", "Added": "",
              "Columns after the step": len(STREET_MAIN), "Reason": f"Final model: {FINAL_TYPE}", "Examples": ""})
steps.append({"Step": k + 1, "Stage": "Result", "What happens": "Final full model", "Removed": "", "Added": len(CYCLIST),
              "Columns after the step": len(STREET_MAIN) + len(CYCLIST), "Reason": "The street model plus the cyclist's age, sex and age group", "Examples": ""})
journey_steps = pd.DataFrame(steps)
show(journey_steps[["Step", "Stage", "What happens", "Removed", "Added", "Columns after the step"]], "The feature journey, step by step:", 0)
show(journey.groupby(["Family group", "Eligible pre-crash input"]).size().unstack(fill_value=0), "\nColumns by family group:", 0)
if running != len(CANDIDATES):
    print(f"Note: the stage 1 counts add up to {running}, the catalog has {len(CANDIDATES)} candidates (check the stage 1 journey file).")

# ============ 10.2 The data file: eligible pre-crash inputs + the outcome ============
DATA_FILE = os.path.join(OUTPUT_FOLDER, "eligible_pre_crash_inputs.csv")
recent_rows = data.iloc[:n]
part = np.empty(n, dtype=object)
part[IDX["fit"]], part[IDX["val"]], part[IDX["test"]] = "fitting", "validation", "test"
export = recent_rows[["collision_id", "accident_dt"] + ELIGIBLE].copy()
export.insert(2, "part", part)
export["casualty_severity"] = recent_rows[OUTCOME_COLUMN].values
export["serious_or_fatal"] = recent_rows["target"].values
export.to_csv(DATA_FILE, index=False)
dictionary = []
for c in export.columns:
    if c in ("collision_id", "accident_dt", "part"):
        role, note = "Identifier / split", {"collision_id": "Collision identifier", "accident_dt": "Date and time of the crash",
                                            "part": "Time split used in the analysis: fitting (oldest 70%), validation (next 15%), test (newest 15%)"}[c]
        fam = ""
    elif c in ("casualty_severity", "serious_or_fatal"):
        role, fam = "Outcome", "Outcome"
        note = "Recorded severity (Slight / Serious / Fatal)" if c == "casualty_severity" else "1 = serious or fatal, 0 = slight (the outcome modelled)"
    else:
        role, fam = "Pre-crash input", family_of(c, "")
        note = "; ".join(x for x in [stage2_of(c)[0], correction_of(c), missing_rule_of(c)] if x)
    dictionary.append({"Column": c, "Role": role, "Family group": FAMILY_GROUP.get(fam, "" if role != "Outcome" else "Outcome"), "Family": fam,
                       "Type": "number" if pd.api.types.is_numeric_dtype(export[c]) else "category",
                       "Missing %": round(export[c].isna().mean() * 100, 2), "Notes": note})
data_dictionary = pd.DataFrame(dictionary)
print(f"\nData file saved: {DATA_FILE} ({len(export):,} crashes x {export.shape[1]} columns: "
      f"{len(ELIGIBLE)} eligible inputs, 2 outcome columns, 3 identifier and split columns).")
print("Values are as used in the analysis (after the stage 2 corrections); empty cells are missing values, not yet filled.")

# ============ 10.3 Settings, decision log and the sheet guide ============
settings_rows = [{"Item": "Final model", "Value": FINAL_TYPE},
                 {"Item": "Feature selection", "Value": FINAL_OPTION},
                 *[{"Item": f"{m}: settings", "Value": json.dumps({k_: v for k_, v in plain(BEST[m]).items() if k_ != "selection"})} for m in MEMBERS],
                 {"Item": "Street features per model", "Value": json.dumps({m: len(MEMBER_STREET[m]) for m in MEMBERS})},
                 {"Item": "Older crashes in training", "Value": f"{'yes' if USE_OLDER else 'no'} ({len(IDX['older']):,} available)"},
                 {"Item": "Seeds (tree models averaged)", "Value": str(SEEDS)},
                 {"Item": "Start date", "Value": START_DATE},
                 {"Item": "Search", "Value": f"{len(search_all)} settings ({N_SETTINGS} per model type), {N_FOLDS} time-ordered folds in the fitting part"},
                 {"Item": "Class weights", "Value": "balanced: both classes weigh the same in total"},
                 {"Item": "Missing values", "Value": f"marker only for families: {sorted(PLACE_FAMILIES)}; pairs and ratios missing when a source is missing"},
                 {"Item": "One-hot encoding", "Value": "min_frequency=100, max_categories=50"},
                 {"Item": "Excluded columns", "Value": json.dumps(EXCLUDED_COLUMNS)},
                 {"Item": "Cut-off", "Value": "best balanced accuracy on validation (fitting-only model), same share flagged on test"},
                 {"Item": "Python / pandas / scikit-learn / xgboost / lightgbm",
                  "Value": f"{platform.python_version()} / {pd.__version__} / {sklearn.__version__} / {xgb.__version__} / {lgb.__version__}"}]
SHEETS["Settings"] = pd.DataFrame(settings_rows)

main_cmp = comparisons[comparisons["Main question"]].iloc[0]
decisions = [
    ("Period", f"Crashes from {START_DATE}", "Injury-based severity recording from November 2016: one definition of 'serious' throughout", "Section 2", ""),
    ("Unit of analysis", "One row per collision, the most seriously injured cyclist", "Each crash counts once", "Stage 1", ""),
    ("Outcome", "Serious or fatal vs slight", "Fatal alone is too rare", "Stage 1", ""),
    ("Time split", "Fitting 70%, validation 15%, test 15%, by date", "Trained on the past, tested on the future; the final model uses 85% (fitting + validation)", "Section 2", ""),
    ("Rows used", "Crashes with a safety score", "All versions compared on the same crashes", "Section 2", ""),
    ("Eligible inputs", f"{len(ELIGIBLE)} columns (see 'Feature journey steps')", "Only information available before the crash about the place, the conditions and the cyclist", "Stage 1 and Section 3", ""),
    ("Excluded columns", ", ".join(EXCLUDED_COLUMNS), "Not properties of the kind of street", "Section 3.1", ""),
    ("Missing values", "Marker only for place features; pairs and ratios missing when a source is missing",
     "Missing police and cyclist details are linked to less serious crashes (recording, not the street)", "Section 3.2", ""),
    ("Junction control", "'Not at junction' when the crash is not at a junction", "The field is empty because there is no junction to control; a real fact about the place", "Section 2", ""),
    ("Feature selection", f"Chosen in the search among {len(OPTIONS)} options: '{FINAL_OPTION}'",
     "Correlation thresholds at standard significance levels (z / sqrt(n)), stability selection and mRMR; selected inside each fold on its training crashes only",
     "Sections 3.3 and 4", "Peng, Long and Ding (2005); Meinshausen and Buhlmann (2010)"),
    ("Model types", ", ".join(MODEL_TYPES), "The best three of eight types in the earlier comparison", "Earlier model comparison", ""),
    ("Reference versions", "Safety score only; cyclist only", "Show what each source of information predicts on its own", "Section 6", ""),
    ("Photo timing", "Tested as one extra version (full model + months between crash and photo), not an eligible street feature",
     "It describes data collection rather than the street and can carry the crash date; the safety score is also checked on crashes with recent photos",
     "Sections 6.1 and 6.3", ""),
    ("Settings search", f"{N_SETTINGS} random settings per model type", "Scored by mean ROC-AUC over 5 time-ordered folds in the fitting part", "Section 4", ""),
    ("Final model", f"{FINAL_TYPE}", "Highest balanced accuracy on the validation part among four candidates: the best setting of each type and their average", "Section 5", ""),
    ("Older crashes (before the start date)", "Used for training" if USE_OLDER else "Not used",
     "Labels put on the injury-based scale with the DfT adjustment; used only if validation balanced accuracy rises", "Section 5.2", "DfT severity adjustment"),
    ("Retraining on fitting + validation", "Exactly the same street features as validated", "Keeps the tested model identical to the validated one", "Section 6", ""),
    ("Cut-off", "Best balanced accuracy", "Balanced accuracy is the measure reported", "Sections 5 and 6", ""),
    ("Main safety question", "Full + safety mean vs full",
     f"Balanced accuracy {main_cmp['BA with %']:.1f}% vs {main_cmp['BA without %']:.1f}% (difference {main_cmp['Difference (points)']:+.2f} points, range {main_cmp['Difference 95% range']})",
     "Section 6", ""),
    ("Feature impact", "SHAP on the test part", "Size and direction of each input's effect", "Section 7", "Lundberg and Lee (2017)"),
    ("Robustness", "Two time periods", "One test period is not enough on its own", "Section 9", ""),
]
SHEETS["Decision log"] = pd.DataFrame(decisions, columns=["Decision", "Choice", "Reason", "Evidence", "Source"])
SHEETS["Feature journey steps"] = journey_steps
SHEETS["Feature journey"] = journey
SHEETS["Data file columns"] = data_dictionary

# ---- feature dictionary of the final models ----
FAMILY_TEXT = {"Crash history counts": "Count of earlier crashes around the crash location", "Earlier severity ratios":
               "How often earlier crashes of the same kind were serious, relative to all crashes at that time (earlier crashes only)",
               "Road feature pairs": "Two police-recorded road fields combined into one category", "Weather readings": "Weather reading before the crash (Open-Meteo archive)",
               "Points of interest": "Distance to, or number of, nearby places", "Road and crash record": "Recorded by the police about the road and the conditions",
               "Calendar": "From the date and time of the crash", "Noise": "Road traffic noise at the location", "Deprivation and high street":
               "Area deprivation (IMD) and high-street location", "Cyclist exposure": "How many cyclists use the location", "Cyclist": "About the cyclist",
               "Safety score": "Perceived safety of the street from the photos (Paper 1 model)", "Photo timing": "Time between the crash and the street photo"}
final_features = list(dict.fromkeys(STREET_MAIN + CYCLIST + SAFETY_COLS + PHOTO))
dict_rows, cat_rows = [], []
for f in final_features:
    fam = family_of(f, "") if f not in PHOTO else "Photo timing"
    is_cat = not pd.api.types.is_numeric_dtype(data[f])
    used = "Full model only" if f in CYCLIST else ("Safety versions" if f in SAFETY_COLS else ("Photo-timing version" if f in PHOTO else
           ", ".join(m for m in MEMBERS if f in MEMBER_STREET[m])))
    dict_rows.append({"Feature (code)": f, "Readable name": readable_feature(f), "Family group": FAMILY_GROUP.get(fam, fam), "Family": fam,
                      "Type": "category" if is_cat else "number", "What it measures": FAMILY_TEXT.get(fam, ""), "Used by": used,
                      "Missing values": missing_rule_of(f) or "Filled with the median, no marker",
                      "Share of SHAP importance % (full model)": round(float(shap_share.get(f, np.nan)), 2) if f in shap_share.index else np.nan})
    if is_cat:
        vc = data[f].iloc[:n].value_counts()
        for v, c in vc.items():
            cat_rows.append({"Feature (code)": f, "Readable feature": readable_feature(f), "Category (code)": v,
                             "Category (readable)": readable_category(v), "Crashes": int(c)})
feature_dictionary = pd.DataFrame(dict_rows)
category_codes = pd.DataFrame(cat_rows)
SHEETS["Feature dictionary"] = feature_dictionary
SHEETS["Category codes"] = category_codes
DICT_PATH = os.path.join(OUTPUT_FOLDER, "feature_dictionary.xlsx")
with pd.ExcelWriter(DICT_PATH, engine="openpyxl") as w:
    for nm_, tb in [("Features", feature_dictionary), ("Category codes", category_codes)]:
        tb.to_excel(w, sheet_name=nm_, index=False)
        ws = w.sheets[nm_]
        ws.freeze_panes = "B2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.font, cell.fill = Font(bold=True), PatternFill("solid", fgColor="DCE6F0")
        for i, col in enumerate(tb.columns, 1):
            ws.column_dimensions[get_column_letter(i)].width = min(70, max(12, int(tb[col].astype(str).str.len().head(300).max() or 10) + 2))
print(f"Feature dictionary saved: {DICT_PATH} ({len(feature_dictionary)} features, {len(category_codes)} category codes)")

# ---- the trained final models, for new predictions without retraining ----
import joblib
saved_models = {"final model": FINAL_TYPE, "members": MEMBERS, "how to predict": "for each member: preprocessor.transform(table[columns]), "
                "average the members' models' predict_proba[:, 1]; the final score is the average over members",
                "cut-offs (share of crashes flagged)": {k_: float(SHARES[k_]) for k_ in ("Street", "Full")}}
for version in ("Street", "Full"):
    saved_models[version] = {m: {k_: RESULTS[(m, version, "fit+val")][k_] for k_ in ("preprocessor", "models", "cols")} for m in MEMBERS}
joblib.dump(saved_models, os.path.join(OUTPUT_FOLDER, "final_models.joblib"))
print("Saved final_models.joblib (street and full models, trained on fitting + validation).")

full_row = test_table.set_index("Version").loc["Full"]
SHEET_GUIDE = {
    "Read me": "This guide: what each sheet contains",
    "Feature journey steps": "How the raw columns became the final inputs, step by step, with counts and reasons",
    "Feature journey": "Every column: family, what happened in stage 1 and stage 2, whether it is an eligible pre-crash input and which models use it. Use the filters in the header row.",
    "Data file columns": "Description of every column in eligible_pre_crash_inputs.csv",
    "Feature dictionary": "Readable name, family and meaning of every feature in the final models (also saved as feature_dictionary.xlsx)",
    "Category codes": "Every category code in the final features and its readable label (for example Cwy = carriageway)",
    "Decision log": "Every decision, its reason and where the evidence is",
    "Settings": "Final model, settings, seeds and package versions",
    "Data and split": "Crashes in the fitting, validation and test parts",
    "Missing correction": "Values set to missing in road feature pairs and earlier-severity ratios",
    "Missing values": "Features with missing values, the serious share when missing or present, and how they are filled",
    "Feature association": "Correlation and mutual information of every street feature with severity (fitting part)",
    "Selection options": "The 10 feature-selection options and how many features each keeps",
    "Search": "Every setting tried in the model search, with its score on each fold",
    "Search best per type": "The best setting of each model type",
    "Search by selection": "The best score of each selection option, per model type",
    "Model choice": "The four candidates on the validation part, and the final model",
    "Older crashes": "The test of adding crashes from before the start date",
    "Models for the paper": "The final model and the three single models on validation and test",
    "Photo timing check": "The safety score on test crashes with recent photos",
    "Validation results": "Every version on the validation part",
    "Test results": "Every version on the test part: balanced accuracy, recall, specificity, ROC-AUC and ranges",
    "Test comparisons": "Balanced accuracy with and without the safety score",
    "Risk tenths": "Share of serious crashes in each tenth of predicted risk",
    "Chart data months": "Crashes and serious share per month (data of chart 01)",
    "Seasons": "Crashes and serious share by season",
    "Chart data ROC": "Points of the ROC curves (data of chart 05)",
    "Top feature severity": "Serious share across the values of the six most important features",
    "SHAP inputs": "Impact and direction of every input (each category separately)",
    "SHAP features": "Impact of every feature (categories added together)",
    "Breakdowns": "Share of crashes and serious or fatal share by borough, road, time, cyclist and area (all crashes from the start date)",
    "Borough map data": "Crashes, serious share, average safety score and predicted risk by borough, as shown on the maps",
    "Time periods": "Results in two other time periods",
}
order = list(SHEET_GUIDE)
SHEETS["Read me"] = pd.DataFrame(
    [{"Sheet": "", "What it contains": f"Stage 2 results, {time.strftime('%Y-%m-%d')}. Final model: {FINAL_TYPE}. "
      f"Full model on the test part: balanced accuracy {full_row['Balanced accuracy %']:.1f}% ({full_row['BA 95% range']}), ROC-AUC {full_row['ROC-AUC']:.3f}."}] +
    [{"Sheet": s, "What it contains": d} for s, d in SHEET_GUIDE.items() if s in SHEETS or s == "Read me"])
SHEETS = {k_: SHEETS[k_] for k_ in order if k_ in SHEETS}

def save_excel_final():
    header = PatternFill("solid", fgColor="DCE6F0")
    with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
        for name, table in SHEETS.items():
            sheet = name[:31]
            table.to_excel(writer, sheet_name=sheet, index=False)
            ws = writer.sheets[sheet]
            ws.freeze_panes = "B2" if name in ("Feature journey", "Data file columns") else "A2"
            if table.shape[0] and table.shape[1]:
                ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font, cell.fill = Font(bold=True), header
                cell.alignment = Alignment(wrap_text=True, vertical="top")
            for i, col in enumerate(table.columns, 1):
                longest = max([len(str(col))] + [len(str(v)) for v in table[col].head(300)])
                ws.column_dimensions[get_column_letter(i)].width = min(70, max(10, longest + 2))
    print(f"Excel saved: {EXCEL_PATH} ({len(SHEETS)} sheets)")
save_excel_final()

predictions = data.iloc[IDX["test"]][["collision_id", "accident_dt", "target"]].reset_index(drop=True)
for name in VERSIONS:
    predictions[name] = pred(name, "fit+val", "test")
predictions.to_csv(os.path.join(OUTPUT_FOLDER, "test_predictions.csv"), index=False)
print(f"Saved: test_predictions.csv; charts: {len([f for f in os.listdir(CHART_FOLDER) if f.endswith('.pdf')])} PDF files")

### The results in plain words

In [ ]:
print(f"Crashes from {START_DATE}: {n:,} (test part {len(yt):,}, {yt.mean() * 100:.1f}% serious or fatal).")
print(f"Search: {len(search_all)} settings over {len(MODEL_TYPES)} model types. Final model: {FINAL_TYPE}, "
      f"feature selection '{FINAL_OPTION}' ({len(STREET_MAIN)} street features).")
tt = test_table.set_index("Version")
for b in ["Street", "Full"]:
    print(f"  {b} model: balanced accuracy {tt.at[b, 'Balanced accuracy %']:.1f}% ({tt.at[b, 'BA 95% range']}), ROC-AUC {tt.at[b, 'ROC-AUC']:.3f}")
print(f"Main safety question (full + safety mean vs full): {main_cmp['BA with %']:.1f}% vs {main_cmp['BA without %']:.1f}%, "
      f"difference {main_cmp['Difference (points)']:+.2f} points ({main_cmp['Difference 95% range']}).")
print(f"Most important input: {shap_table.at[0, 'Input']} ({shap_table.at[0, 'Direction']}).")
print(f"Time periods, full model balanced accuracy: " + ", ".join(f"{r['Test period']}: {r['Full BA %']:.1f}%" for _, r in periods.iterrows()))
print(f"\nWhole notebook: {(time.time() - notebook_start) / 60:.0f} min")